# LLM Fine-Tuning Deep Dive: Data-Based vs Parameter-Based Techniques

## The brief: Riverside House needs an in-house AI, not an API call

**Riverside House** is a small publishing firm. Everything under [`content/`](content/) is their
**unpublished, proprietary manuscript catalog** -- seven complete novels spanning sci-fi, fantasy,
mystery, historical fiction, cyberpunk, horror, and literary fiction, still under contract, still
unreleased. That's exactly why nobody at Riverside is allowed to paste chapters into a public chatbot
API: the moment draft manuscripts leave the building, the confidentiality clause is broken. Whatever
model they use has to be trained and run **in-house**, on hardware they already own -- a laptop, not a
GPU cluster, and no data ever leaves it.

Riverside's ask has two parts:

1. **An editing assistant** for the ghostwriters and continuity editors -- prompt it with a scene and
   get back a continuation that actually remembers who Aria Voss is and what the Meridian's Promise
   is, follows a direct instruction instead of rambling forever, and reads the way their editors
   _actually_ prefer.
2. **A knowledge base for the rest of the company** -- marketing, licensing, and new hires who need
   answers like "who are the six founding families in the mystery novel?" without reading 197 chapters
   or, worse, guessing.

Today, every one of those people either re-reads old chapters by hand or asks a colleague. That's the
gap this notebook closes -- and by the end we have to pick **one model to actually deploy**, backed by
more than "it read fine to me."

**Goal:** fine-tune a small-but-capable base model (`gpt2-medium`, ~355M parameters) on Riverside's
proprietary catalog -- **seven complete novels** (~619,000 words / ~3.0 MB total, entirely inside
[`content/`](content/)) -- so it learns their characters, invented terminology, and prose style across
genres, demonstrating **every major axis of fine-tuning** along the way:

| Step | Concept                            | Riverside's question                                                                         |
| ---- | ---------------------------------- | -------------------------------------------------------------------------------------------- |
| 1    | Continued pretraining              | Does it even know our characters and world exist?                                            |
| 2    | Instruction tuning (SFT)           | Does it follow a "continue this scene" / "answer this question" request instead of rambling? |
| 3    | Preference alignment (DPO)         | Does it write and answer the way our editors actually prefer?                                |
| 4    | Full fine-tuning                   | Best quality -- but what does it cost on a laptop?                                           |
| 5    | Partial freezing                   | A cheaper middle ground -- how much quality do we give up?                                   |
| 6    | LoRA                               | The cheapest option -- is it good enough to ship?                                            |
| 7    | Ablation study                     | What breaks if the deadline forces us to skip a stage?                                       |
| 8    | Head-to-head + held-out perplexity | Which model do we actually deploy in-house?                                                  |

| Axis                | Question it answers                         | Techniques covered here                                                                           |
| ------------------- | ------------------------------------------- | ------------------------------------------------------------------------------------------------- |
| **Data-based**      | _What objective/data teaches the behavior?_ | Non-instructional (continued pretraining), Instructional (supervised), Preference alignment (DPO) |
| **Parameter-based** | _How many/which weights are updated?_       | Full fine-tuning, Partial (layer freezing), Parameter-efficient (LoRA)                            |

## Corpus (Riverside House's proprietary manuscripts)

- **Location:** [`content/`](content/) -- **7 original, unpublished novels** across diverse genres
  (sci-fi, fantasy, mystery, historical, cyberpunk, horror, literary), totaling **197 chapters**
  (~619,000 words / ~3.0 MB). This directory _is_ Riverside's confidential catalog for the purposes
  of this notebook. See [`content/README.md`](content/README.md) for the full breakdown and
  individual synopses.
- **Genres available:**
  - Sci-fi: _The Weight of Distant Light_ (generation ship, 40 chapters)
  - Fantasy: _The Tidebound Accord_ (epic quest, 33 chapters)
  - Mystery: _The Cartographer's Cipher_ (noir detective, 21 chapters)
  - Historical: _The Silk Merchant's Daughter_ (Tang Dynasty, 23 chapters)
  - Cyberpunk: _Neural Drift_ (memory broker conspiracy, 24 chapters)
  - Horror: _The Hollow Beneath_ (gothic/cosmic, 28 chapters)
  - Literary: _The Weight of Tides_ (marine biology first contact, 28 chapters)
- **Scale note:** Training cells sample a subset (`max_chapters` per genre) by default for fast CPU
  demos. Pass larger limits or `genres=None` to train on the full corpus.

## Setup

Run `setup.ps1` once to create a `.venv` and register the `llm-tuning` Jupyter kernel, then select
that kernel for this notebook.


In [ ]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2-medium"  # ~355M params, real pretrained weights, still CPU-trainable but far better suited to actually absorbing a domain corpus than distilgpt2

# Resolve the notebook's own directory (content/ lives next to this notebook). VS Code's Jupyter
# kernels run with cwd = workspace root, not the notebook's folder, so __vsc_ipynb_file__ (which
# VS Code injects) is the reliable way to find it; __file__ covers plain .py execution.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():
    # Fallback for kernels whose cwd is the repo root instead of this notebook's own folder
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel directory mappings (keys are shorthand aliases, values are actual directory names)
NOVELS = {
    "scifi":      "the-weight-of-distant-light",   # 40 chapters
    "fantasy":    "the-tidebound-accord",           # 33 chapters
    "mystery":    "the-cartographers-cipher",       # 21 chapters
    "historical": "the-silk-merchants-daughter",    # 23 chapters
    "cyberpunk":  "neural-drift",                   # 24 chapters
    "horror":     "the-hollow-beneath",             # 28 chapters
    "literary":   "the-weight-of-tides",            # 28 chapters
}


def load_corpus_paragraphs(novels=None, max_chapters=4, min_len=200):
    """Load paragraphs from the multi-novel corpus for quick CPU demos.

    Args:
        novels: List of novel aliases to load (e.g., ["scifi", "fantasy"]), or None to
                load all. Available aliases: "scifi", "fantasy", "mystery", "historical",
                "cyberpunk", "horror", "literary" (mapped to directory names).
        max_chapters: Max chapters to load per novel (keeps CPU training fast).
        min_len: Skip paragraphs shorter than this many characters.

    Returns:
        List of paragraph strings from all requested novels.
    """
    if novels is None:
        novels = list(NOVELS.keys())  # load all by default

    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            print(f"Warning: unknown novel alias '{alias}', skipping")
            continue

        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            print(f"Warning: directory {novel_path} not found, skipping")
            continue

        chapter_files = sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)

    return paragraphs


def tokenize_causal(examples, tokenizer, max_length=128):
    """Standard next-token-prediction tokenization: labels = input_ids, with padding
    positions masked out (-100) so the loss ignores them."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


# Sample from 4 genres to show multi-genre paragraph diversity
sample_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery", "horror"], max_chapters=2
)
print(
    f"Loaded {len(sample_paragraphs)} sample paragraphs from 4 novels. First one:\n"
)
print(sample_paragraphs[20][:421], "...")

In [ ]:
# Visualization imports for intuition building
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
from IPython.display import display, HTML
import warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Visualization libraries loaded.")

## Why Fine-Tuning? The Three-Gap Problem

A pretrained LLM (like GPT-2, LLaMA, or Mistral) has learned language from billions of tokens of web
text, books, and code. This gives it strong **general fluency** and **broad world knowledge**. That's
exactly why `gpt2-medium` is a reasonable starting point for Riverside House's assistant -- it can
already write fluent English. But for Riverside's specific job, it has three concrete gaps:

### Gap 1: Domain Knowledge Gap

**Problem:** The model has never seen Riverside's specific vocabulary, characters, facts, or house
style.

**Example with our corpus:**

- **An editor asks:** `"Who is Aria Voss?"`
- **Base model:** `"Aria Voss is a... [makes up something generic or says 'I don't know']"`
- **After fine-tuning:** `"Aria Voss is the Hold systems technician aboard the Meridian's Promise 
generation ship..."`

**Solution:** Continued pretraining on Riverside's catalog.

---

### Gap 2: Behavior Gap

**Problem:** A raw pretrained model just continues text. It doesn't know how to follow instructions,
answer questions directly, or stop when it should -- which is a problem the moment a ghostwriter wants
to _ask_ something instead of just seeding a paragraph.

**Example:**

- **An editor asks:** `"List the five tides in the Tidebound Accord."`
- **After domain pretraining:** `"List the five tides in the Tidebound Accord. This question has 
puzzled scholars for millennia. Some say there are actually six tides, while others..."` (rambles
  forever)
- **After instruction tuning:** `"The five tides are: water, wind, stone, flame, and void."`

**Solution:** Instruction tuning (supervised fine-tuning) on (prompt, completion) pairs.

---

### Gap 3: Preference Gap

**Problem:** Even an instruction-following model may produce outputs that are technically correct but
not what Riverside's editors actually prefer (too verbose, wrong tone, unhelpful focus) -- and "reads
worse than a human editor would tolerate" is exactly the kind of gap that kills adoption of an internal
tool, even when it "technically works."

**Example:**

- **An editor asks:** `"Explain quantum entanglement simply."`
- **After instruction tuning:** `"Quantum entanglement is a phenomenon in quantum mechanics wherein 
the quantum states of two or more particles become interdependent such that the state of one cannot 
be fully described without reference to the others, even when separated by large distances..."` (10
  more paragraphs of jargon)
- **After preference alignment:** `"Quantum entanglement means two particles become connected so 
measuring one instantly affects the other, even across vast distances."`

**Solution:** Preference alignment (RLHF or DPO) using human preference data.

---

### The Different Paths towards the same journey - ILearned Intelligence over custom data

The choices of the pre-training technique depends on the most crucial gap

```
Pretrained base ---> Continued pretraining ---> Instruction tuning ---> Preference alignment ---> Production model
    (Gap 0)               (Closes Gap 1)              (Closes Gap 2)            (Closes Gap 3)
```

At each stage, you also choose **how many parameters to update** (full fine-tuning vs. freezing vs.
LoRA) -- which, for Riverside House, is really a budget question: they have a laptop, not a GPU
cluster. We'll explore that trade-off after demonstrating the data-based journey.

---

## Baseline: What Does the Un-Tuned Model Know?

Before fine-tuning, let's see what `gpt2-medium` (pretrained on generic web text) produces when
prompted with a scenario from Riverside's sci-fi novel. Since it has never seen this story, expect a
fluent but generic, off-world continuation -- this is the starting point Riverside's editors are stuck
with today.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus


def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base GPT-2, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]          # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,                            # stochastic → varied output
            top_p=0.9,                                 # nucleus sampling: top 90% mass
            temperature=0.8,                           # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )
    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return completion if completion else "[model stopped immediately — sampled EOS as first token]"


print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")
print(f"Completion: {generate(base_model, PROMPT)}")

### Code Walkthrough: Setup Cell

**What just ran — three building blocks used throughout this entire notebook:**

---

**1. `tokenizer.pad_token = tokenizer.eos_token`**

GPT-2 was pretrained on sequences of a fixed length with no padding token in its vocabulary. When the `Trainer` batches examples of different lengths it needs a pad token to fill the shorter sequences. Setting it to `eos_token` (ID 50256) is the standard convention — it tells the tokenizer "treat end-of-sequence as padding." The matching `labels=-100` mask in `tokenize_causal` (in the setup cell above) ensures the model is _never_ penalized for "predicting" those padding positions.

---

**2. `AutoModelForCausalLM.from_pretrained(MODEL_NAME)`**

This loads the full `gpt2-medium` checkpoint: 355 million parameters, 24 transformer blocks, hidden size 1024. The `.to(device)` call moves all weight tensors to CPU (or GPU if available). The `AutoModel` family is generic — swap `MODEL_NAME` for `"meta-llama/Llama-3.1-8B"` and the rest of the loading code adapts automatically.

---

**3. `generate(model, prompt, max_new_tokens=60)` — decoding strategy**

A thin wrapper around HuggingFace's `model.generate()` with three key choices:

| Parameter         | Value               | Effect                                                   |
| ----------------- | ------------------- | -------------------------------------------------------- |
| `do_sample=True`  | Stochastic decoding | Avoids repetitive, deterministic greedy output           |
| `top_p=0.9`       | Nucleus sampling    | Considers only tokens whose cumulative probability ≥ 90% |
| `temperature=0.8` | Slight smoothing    | Reduces "safe" word dominance without full randomness    |

The function returns **only the newly generated tokens** (after slicing off the prompt), so every `print(generate(...))` call shows the model's actual continuation — not the prompt echoed back at you.

> **PyTorch shape note:** `out = model.generate(...)` returns a tensor of shape `(batch=1, total_len)` where `total_len = prompt_len + max_new_tokens`. We slice `out[0][prompt_len:]` to get just the new tokens.


## Test Prompts for Validating Fine-Tuning

Use these prompts to test whether fine-tuning successfully absorbed domain-specific knowledge from
the seven-novel corpus. A well-tuned model should recognize characters, settings, and continue
narratives in the appropriate style. The baseline model (pretrained only) should produce generic,
off-topic continuations.

### Character & Setting Recognition Tests

**Sci-Fi (The Weight of Distant Light):**

- `"Aria Voss checked the Meridian's Promise status panel and"`
- `"The Keeper's consciousness flickered through node seventeen as"`
- `"In the Under-Hold, Nyla Kade whispered about the prime number signal from"`

**Fantasy (The Tidebound Accord):**

- `"Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as"`
- `"The ancient pillars rose from the Abyssal Rift while Davin Shale"`
- `"The Hollow King's followers, called the Hollowed,"`

**Mystery (The Cartographer's Cipher):**

- `"Elena Voss studied the 1879 survey map and realized the Ashmont Trust"`
- `"Detective Chen examined Adelaide Thorne's body and found the message: 'the foundation must hold'"`
- `"The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—"`

**Historical (The Silk Merchant's Daughter):**

- `"Wei Lian's jade phoenix pendant caught the morning light in Chang'an as"`
- `"Zhang Ming, the jinshi degree holder, wrote in his letter"`
- `"In the Eastern Market, the Wei family silk compound"`

**Cyberpunk (Neural Drift):**

- `"Kai Chen adjusted the neurorig and prepared to extract the memory backup from"`
- `"In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift"`
- `"Victor Tang's consciousness transfer protocol failed when"`

**Horror (The Hollow Beneath):**

- `"Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor"`
- `"The hollow beneath the house breathed, and the entity in the limestone caves"`
- `"Margot found Thaddeus Blackwood's journal warning: never descend past the second chamber"`

**Literary (The Weight of Tides):**

- `"Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about"`
- `"In Willowport, the underwater object near Whitehead Island caused"`
- `"The lobster traps came up bent, and the water temperature dropped fifteen degrees when"`

### Genre Style Continuation Tests

**Sci-Fi narrative momentum:**

- `"Two hundred and fourteen years after the Meridian's Promise left Earth,"`

**Fantasy elemental magic:**

- `"The tide-weavers gathered at Deepwater Crossing as the fifth tide, the void tide,"`

**Mystery noir atmosphere:**

- `"The rain-slicked streets of Ashmont Bay hid secrets from 1879, and Elena Voss"`

**Historical detail & restraint:**

- `"The silk road brought more than trade goods to Tang Dynasty Chang'an—it brought"`

**Cyberpunk tech-noir:**

- `"Memory extraction left traces, neural signatures that couldn't be scrubbed, and Kai Chen"`

**Gothic horror tension:**

- `"The house chose its inhabitants through grief, calling them when they were most vulnerable, and"`

**Literary introspection:**

- `"The ocean held its own memory, deeper and older than human documentation, and Claire"`

### Cross-Novel Vocabulary Tests

These should work across multiple genres if fine-tuning absorbed the corpus style:

- `"The weight of distant"` (tests sci-fi novel phrase bleed)
- `"The tidebound"` (tests fantasy terminology)
- `"permanent removal"` (tests mystery euphemism)
- `"steel in your spine, even if you must hide it beneath"` (tests historical voice)
- `"neural backup"` (tests cyberpunk jargon)
- `"the hollow"` (tests horror atmospheric language)
- `"The water's wrong"` (tests literary marine biology voice)


In [ ]:
# Automated test runner: compare baseline vs fine-tuned on corpus-specific prompts
TEST_PROMPTS = {
    # Sci-fi — The Weight of Distant Light
    "scifi_character": "Aria Voss checked the Meridian's Promise status panel and",
    "scifi_keeper": "The Keeper's consciousness flickered through node seventeen as",
    # Fantasy — The Tidebound Accord
    "fantasy_magic": "Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as",
    "fantasy_hollow_king": "The Hollow King's followers, called the Hollowed, began to gather when",
    # Mystery — The Cartographer's Cipher
    "mystery_conspiracy": "The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—",
    "mystery_elena": "Elena Voss studied the 1879 survey map and realized the Ashmont Trust",
    # Historical — The Silk Merchant's Daughter
    "historical_setting": "Wei Lian's jade phoenix pendant caught the morning light in Chang'an as",
    "historical_silk_road": "The delegation crossed the Taklamakan desert and Wei Lian noted in her ledger",
    # Cyberpunk — Neural Drift
    "cyberpunk_tech": "Kai Chen adjusted the neurorig and prepared to extract the memory backup from",
    "cyberpunk_project": "In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift",
    # Horror — The Hollow Beneath
    "horror_atmosphere": "Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor",
    "horror_chambers": "The tenth chamber of the Hollow pulsed with a light that had no source, and Eleanor",
    # Literary — The Weight of Tides
    "literary_marine": "Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about",
    "literary_contact": "The Observer surfaced near Whitehead Island and Claire understood for the first time that",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run all test prompts and return results dict for comparison."""
    results = {}
    for key, prompt in test_prompts.items():
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)
    return results


# Run baseline tests (will show generic, off-corpus continuations)
print(
    "=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===\n"
)
baseline_results = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(output[:200] + "...\n")

Notice the output has no awareness of Aria Voss (from _The Weight of Distant Light_), the _Meridian's
Promise_, the Lantern, or any of the other characters/worlds across the seven novels -- it is fluent
English but a generic, unrelated continuation. This is exactly the gap fine-tuning closes.


### Understanding One Training Step: A Concrete Example

Before we start training, let's walk through **exactly what happens** during a single training step of
continued pretraining, using a real paragraph and the real tokenizer -- the code cell right after this
one runs the actual numbers below through `gpt2-medium` instead of making them up.

**Input:** A paragraph from our sci-fi corpus:

> "Aria Voss stared at the signal counting itself out in prime numbers and felt the weight of two
> centuries press against her ribs."

**Step 1: Tokenization**

The tokenizer converts text → integer IDs, then pads (or truncates) to `max_length=128` so every
example in a batch is the same shape.

**Step 2: Create Labels for Causal LM -- this is the mask layout**

For continued pretraining, the task is **next-token prediction**, and the label at every position is
just the input shifted one to the left:

```
Input IDs: [t0,  t1,  t2,  t3,  ...]
Labels:    [t1,  t2,  t3,  t4,  ...]   # shifted left: "given t0, predict t1", etc.
```

The only masking that happens here is on **padding**: any position past the real text is set to
`-100` in `labels`, so the loss ignores it. Every real token position is active. The graphic below
shows this exact split (real tokens vs. padding) for our example paragraph -- contrast it with the
instruction-tuning mask layout later in the notebook, where the **prompt** is also masked out, not
just the padding.

**Step 3: Forward Pass (Model Prediction)**

The model processes the input and outputs logits (unnormalized scores) for every token position:

```
Logits shape: (batch_size=1, seq_len=128, vocab_size=50257)
```

These are **raw scores**, not probabilities yet.

**Step 4: Compute Loss (Cross-Entropy)**

For each real (non-padding) position, we:

1. Convert logits → probabilities (softmax)
2. Look up the probability assigned to the **correct** next token (the label)
3. Compute negative log probability (this is the loss for that position)
4. Average the per-position losses (ignoring masked/padding positions)

**Step 5: Backpropagation**

Compute gradients: `∂Loss/∂W` for every trainable parameter W in the model.

- **Full fine-tuning:** gradients flow to every parameter (~355M for `gpt2-medium`)
- **Partial freezing:** only the unfrozen last few blocks + head get real gradients
- **LoRA:** only the small adapter matrices get gradients (well under 1% of all parameters)

**Step 6: Optimizer Update**

Update weights in the direction that reduces loss:

```
W_new = W_old - learning_rate × gradient
```

**Step 7: Repeat**

Repeated across many steps, these tiny weight nudges accumulate into **learning**: the model becomes
better at predicting tokens that appear in our domain corpus.

---

**Key Takeaways:**


In [ ]:
# Anatomy of one training step -- computed for REAL on gpt2-medium with the example sentence above
# (not fabricated numbers: this actually tokenizes, forward-passes, and backprops through base_model)
import torch.nn.functional as F

example_text = (
    "Aria Voss stared at the signal counting itself out in prime numbers and felt the "
    "weight of two centuries press against her ribs."
)

enc = tokenizer(
    example_text,
    truncation=True,
    max_length=128,
    padding="max_length",
    return_tensors="pt",
)
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)
real_len = int(attention_mask.sum().item())  # number of real (non-padding) tokens

labels = input_ids.clone()
labels[attention_mask == 0] = -100  # mask padding, exactly like tokenize_causal()

base_model.train()  # need gradients for this one illustrative pass; restored to eval() below
base_model.zero_grad()
outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
step_loss = outputs.loss
step_loss.backward()

# Real per-position loss for the first few real (non-padding) tokens
shift_logits = outputs.logits[0, :-1, :]
shift_labels = labels[0, 1:]
per_token_loss = F.cross_entropy(
    shift_logits, shift_labels, reduction="none", ignore_index=-100
)
positions_to_show = min(8, real_len - 1)
losses_per_pos = per_token_loss[:positions_to_show].detach().cpu().numpy()

# Real gradient magnitude per transformer block (shows actual gradient flow, not a fabricated curve)
n_blocks = base_model.config.n_layer
block_grad_norms = []
for i in range(n_blocks):
    block_params = [
        p
        for n, p in base_model.named_parameters()
        if f"h.{i}." in n and p.grad is not None
    ]
    norm = (
        torch.norm(torch.stack([p.grad.norm() for p in block_params])).item()
        if block_params
        else 0.0
    )
    block_grad_norms.append(norm)

# Real weight update on one real parameter, using the actual computed gradient.
# LR=5e-5 times a small gradient is often ~1e-8 -- too small to see at 6 decimal places, so we
# print W_old/W_new AND the delta in scientific notation so the update is actually visible.
sample_name, sample_param = next(
    (n, p)
    for n, p in base_model.named_parameters()
    if p.grad is not None and p.dim() == 2
)
old_weight = sample_param.data.flatten()[0].item()
sample_grad = sample_param.grad.flatten()[0].item()
lr = 5e-5
weight_delta = -lr * sample_grad
new_weight = old_weight + weight_delta

base_model.zero_grad()
base_model.eval()  # leave base_model exactly as it was for the rest of the notebook

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Anatomy of One Training Step (Continued Pretraining) -- real numbers from gpt2-medium",
    fontsize=13,
    fontweight="bold",
)

# Step 1: Tokenization
ax1 = axes[0, 0]
ax1.text(
    0.5,
    0.88,
    "Step 1: Tokenization",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax1.transAxes,
)
ax1.text(
    0.5,
    0.45,
    f'"{example_text[:38]}..."\n\u2193\n{input_ids[0, :8].tolist()} ...',
    ha="center",
    va="center",
    fontsize=9,
    transform=ax1.transAxes,
    bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
)
ax1.axis("off")

# Step 2: The mask layout (real tokens vs. padding) -- the actual answer to "how is the mask laid out"
ax2 = axes[0, 1]
mask_row = attention_mask[0].cpu().numpy().reshape(1, -1)
ax2.imshow(
    mask_row, cmap="Greens", aspect="auto", vmin=0, vmax=1, extent=[0, 128, 0, 1]
)
ax2.axvline(real_len, color="red", linestyle="--", linewidth=1.5)
ax2.set_yticks([])
ax2.set_xlabel("Token position (0-128)", fontsize=8)
ax2.set_title(
    f"Step 2: Mask Layout\n{real_len} real tokens (green, active) +\n"
    f"{128 - real_len} padding (white, labels=-100)",
    fontsize=9,
    fontweight="bold",
)

# Step 3: Forward pass -- real logits
ax3 = axes[0, 2]
real_logits = outputs.logits[0, :positions_to_show, :50].detach().cpu().numpy()
ax3.imshow(real_logits, cmap="RdYlBu_r", aspect="auto")
ax3.set_xlabel("Vocab (first 50 ids)", fontsize=8)
ax3.set_ylabel("Position", fontsize=8)
ax3.set_title(
    "Step 3: Forward Pass\nreal logits, first tokens", fontsize=9, fontweight="bold"
)
ax3.set_xticks([])
ax3.set_yticks([])

# Step 4: Real per-position loss
ax4 = axes[1, 0]
positions = np.arange(len(losses_per_pos))
ax4.bar(positions, losses_per_pos, color="coral", alpha=0.8, width=0.6)
ax4.axhline(
    losses_per_pos.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Mean (shown)",
)
ax4.set_xlabel("Token position", fontsize=9)
ax4.set_ylabel("Loss", fontsize=9)
ax4.set_title(
    f"Step 4: Compute Loss\nfull-sequence avg = {step_loss.item():.3f}",
    fontsize=9,
    fontweight="bold",
)
ax4.legend(fontsize=8)

# Step 5: Real gradient magnitude per block
ax5 = axes[1, 1]
tick_stride = max(1, n_blocks // 8)
ax5.barh(np.arange(n_blocks), block_grad_norms, color="purple", alpha=0.7)
ax5.set_yticks(np.arange(0, n_blocks, tick_stride))
ax5.set_yticklabels([f"Block {i}" for i in range(0, n_blocks, tick_stride)], fontsize=8)
ax5.set_xlabel("Real gradient norm", fontsize=9)
ax5.set_title(
    "Step 5: Backpropagation\nactual per-block gradient norm",
    fontsize=9,
    fontweight="bold",
)
ax5.invert_yaxis()

# Step 6: Real weight update -- shown at enough precision to actually see the nudge
ax6 = axes[1, 2]
ax6.text(
    0.5,
    0.88,
    "Step 6: Update Weights",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.58,
    f"W_old = {old_weight:.6f}\ngradient = {sample_grad:.3e}\nLR = {lr:.0e}",
    ha="center",
    va="center",
    fontsize=10,
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.28,
    f"\u0394W = {weight_delta:+.3e}\nW_new = {new_weight:.6f}",
    ha="center",
    va="center",
    fontsize=11,
    transform=ax6.transAxes,
    color="green",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="lightgreen", alpha=0.5),
)
ax6.axis("off")

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"\n{'=' * 80}")
print(
    "Training Step Summary (all numbers above are real, from one forward+backward pass):"
)
print(f"{'=' * 80}")
print(f"1. Tokenize: {real_len} real tokens + {128 - real_len} padding tokens")
print("2. Mask layout: labels = input shifted left; padding positions set to -100")
print(f"3. Forward pass: logits shape {tuple(outputs.logits.shape)}")
print(f"4. Loss (avg over real tokens only): {step_loss.item():.3f}")
print(f"5. Backprop: gradients computed for all {n_blocks} transformer blocks")
print(
    f"6. Update: W_new = W_old + \u0394W, where \u0394W = -lr * gradient = {weight_delta:+.3e}"
)
print(
    "   That's why fine-tuning needs many steps: each one nudges a weight by a fraction of a "
    "percent, and Riverside's assistant only 'learns' after thousands of these tiny nudges add up."
)
print(f"{'=' * 80}")

---

## The Fine-Tuning Journey: A Problem-Solution Narrative

Rather than a flat taxonomy, fine-tuning is best understood as a **journey where each technique 
solves a problem left by the previous one**:

```mermaid
flowchart TD
    A[Pretrained Base Model] -->|Problem: Doesn't know your domain| B[Solution: Continued Pretraining]
    B -->|Problem: Continues text, won't follow instructions| C[Solution: Instruction Tuning SFT]
    C -->|Problem: Outputs aren't what humans prefer| D[Solution: Preference Alignment DPO/RLHF]
    
    style A fill:#e1f5ff
    style B fill:#b3e5fc
    style C fill:#81d4fa
    style D fill:#4fc3f7
```

At each stage, you can choose **how many parameters to update**:

| Approach | Trade-off | Use when |
|----------|-----------|----------|
| **Full fine-tuning** (100% params) | Max quality, max cost | Small models, abundant compute |
| **Partial freezing** (10-30% params) | Middle ground | Limited compute budget |
| **LoRA** (well under 1% params) | Min cost, swappable adapters | Most production scenarios |

**This notebook demonstrates:**
- All 3 data-based stages (continued pretraining, instruction tuning, preference alignment)
- All 3 parameter-based approaches (full, partial, LoRA)
- **Not covered** (mentioned for completeness): PPO-based RLHF, adapters, prefix tuning, QLoRA

---


## Concept 1 (Data-Based): Non-Instructional Fine-Tuning (Continued Pretraining)

**Riverside's question for this section:** does the model even know our characters and world exist
yet? Nothing downstream matters if it can't recognize "Aria Voss" or "the Meridian's Promise."

**What it is:** keep training with the exact same objective used for the original pretraining --
next-token prediction -- but on your own raw, unlabeled domain text instead of general web text. No
prompts, no "instructions", no labeled pairs: just plain paragraphs. This is often called _continued
pretraining_ or _domain-adaptive pretraining (DAPT)_.

**When to use it:** you have a pile of domain text (support tickets, legal filings, a publisher's back
catalog...) and you want the model to _absorb_ its vocabulary, facts, and style before you ever teach
it to follow instructions.

**Pros**

- Cheapest data to acquire -- no labeling/annotation needed, just clean text.
- Great at absorbing vocabulary, entities, and stylistic quirks (character names, invented
  terminology...).
- Simple training loop -- identical to pretraining (`labels = input_ids`).

**Cons**

- Does **not** teach the model to follow instructions or hold a conversation -- it only gets better
  at _continuing_ text like your domain text.
- Risk of shallow memorization instead of generalization if the corpus is small or repetitive.
- Risk of **catastrophic forgetting** of general-purpose ability if trained too long/aggressively.

Below we run this on a sample of chapters from Riverside's catalog, updating **all** of
`gpt2-medium`'s parameters (full fine-tuning -- more on that axis further down).


In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

non_inst_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery"], max_chapters=3
)
print(
    f"Loaded {len(non_inst_paragraphs)} paragraphs from 3 novels for continued-pretraining demo"
)

non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})
non_inst_tokenized = non_inst_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

full_ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

training_args_full = TrainingArguments(
    output_dir="./checkpoints/non-instruction-full",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle; raise further for real runs
    logging_steps=10,
    save_strategy="no",
    learning_rate=5e-5,
    report_to="none",
)

trainer_full = Trainer(
    model=full_ft_model, args=training_args_full, train_dataset=non_inst_tokenized
)
trainer_full.train()
full_ft_model.save_pretrained("./checkpoints/non-instruction-full")
print("Saved continued-pretraining (full fine-tune) checkpoint.")

### Cracking Open Full Fine-Tuning: Which Blocks Actually Moved?

Full fine-tuning unfreezes every parameter, but that doesn't mean every block changes by the same
amount. Let's measure it directly: compare `full_ft_model`'s real weights, block by block, against
`base_model` -- the untouched pretrained checkpoint we've never trained. This is the same "crack it
open" treatment LoRA got earlier, applied to full fine-tuning.


In [ ]:
# Real per-block weight-delta norms: full_ft_model (after training) vs. base_model (never trained)
block_weight_deltas = []
base_state = dict(base_model.named_parameters())
for i in range(base_model.config.n_layer):
    delta_norm_sq = 0.0
    for name, p in full_ft_model.named_parameters():
        if f"h.{i}." in name:
            delta = p.data - base_state[name].data
            delta_norm_sq += delta.norm().item() ** 2
    block_weight_deltas.append(delta_norm_sq**0.5)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    range(len(block_weight_deltas)), block_weight_deltas, color="steelblue", alpha=0.85
)
ax.set_xlabel("Transformer block")
ax.set_ylabel("||W_after - W_before|| (real)")
ax.set_title(
    "Full Fine-Tuning: Real Per-Block Weight Movement (full_ft_model vs. untouched base_model)",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(
    f"Every block moved (min \u0394 = {min(block_weight_deltas):.3f}, "
    f"max \u0394 = {max(block_weight_deltas):.3f}) -- full fine-tuning really does touch the whole "
    f"model, which is exactly why it's the most expensive option and the one most likely to nudge "
    f"Riverside's assistant away from plain English while it learns the catalog."
)

# Memory cleanup: full_ft_model's job is done -- non_instruct_ckpt (loaded from disk further down)
# takes over its role for every later comparison. On a laptop with a handful of gpt2-medium-sized
# models in memory at once, freeing this one now is the difference between "fits" and "swaps to disk."
import gc

del full_ft_model
gc.collect()
print(
    "Freed full_ft_model from memory (its checkpoint is saved to disk; non_instruct_ckpt reloads it later)."
)

### Visualizing Training Progress: Loss Curves

After training completes, let's look at what actually happened -- the real per-step loss recorded by
the `Trainer` above, not an idealized illustration. Textbook loss curves are smooth; a 60-step,
batch-size-2 CPU demo on a fresh model is usually much noisier, and that's worth seeing honestly.

**What to look for:**

1. **Downward trend:** Loss should decrease on average (model is learning), even if noisy step-to-step
2. **Convergence:** Loss should stop trending strongly downward by the end (not still falling fast)
3. **Magnitude:** Lower loss = better fit to domain text (but watch for overfitting on tiny corpora!)

**Reading a noisy real curve:** with only 6 logged points and batch_size=2, a single unusually easy or
hard paragraph can swing the reported loss by ±0.3 or more. Don't over-interpret small wiggles -- look
at the overall direction across all points, and compare against the other techniques' real curves
later in the notebook (instruction tuning, partial freezing, LoRA continued pretraining) to see which
setup is converging fastest for the same step budget.


In [ ]:
# Visualize the REAL loss curve from the continued-pretraining run above (trainer_full),
# not a fabricated "typical" curve -- this is exactly what your training just did.
def extract_loss_history(trainer):
    return [
        (entry["step"], entry["loss"])
        for entry in trainer.state.log_history
        if "loss" in entry
    ]


full_ft_history = extract_loss_history(trainer_full)
steps, losses = zip(*full_ft_history)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    steps,
    losses,
    marker="o",
    linewidth=2,
    markersize=7,
    color="green",
    label="Training loss",
)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title(
    f"Continued Pretraining (Full FT): real loss log ({losses[0]:.2f} \u2192 {losses[-1]:.2f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n{'=' * 70}")
print("Reading this REAL loss curve (not an idealized one):")
print(f"{'=' * 70}")
print(f"  Logged steps: {list(steps)}")
print(f"  Logged losses: {[round(l, 3) for l in losses]}")
print(f"  First -> last: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"{'=' * 70}")
print("What to look for:")
print(
    "  • A clean, monotonic plateau like a textbook figure is the exception, not the rule --"
)
print(
    "    especially at batch_size=2 with only a handful of steps, loss is dominated by"
)
print("    per-batch noise (which paragraph happened to be in this batch) more than by")
print("    the underlying trend.")
print(
    "  • If the trend is flat/noisy rather than decreasing: raise max_steps, increase the"
)
print(
    "    batch size, or train on more paragraphs so the trend has room to dominate the noise."
)
print(
    "  • Compare this to the loss curves for instruction tuning, partial freezing, and LoRA"
)
print(
    "    continued pretraining further down -- they were all recorded the same real way."
)
print(f"{'=' * 70}")

### Common Pitfalls: Continued Pretraining

**Pitfall #1: Catastrophic Forgetting**

**Bad:** Train for 10,000 steps on a tiny 50KB domain corpus  
**Good:** Train for 25-100 steps, then validate on general tasks (e.g., "The capital of France is...")

**Why it happens:** The model "overwrites" its general language knowledge with domain-specific patterns.

**How to avoid:**

- Keep training steps low initially (start with 25-50)
- Use a validation set with both domain AND general questions
- Watch for nonsense on general prompts (sign of forgetting)

---

**Pitfall #2: Shallow Memorization**

**Bad:** Tiny corpus (5KB), repeated 100 times → model memorizes exact phrases  
**Good:** Diverse corpus (500KB+) with varied writing styles

**How to detect:**

- Model completes prompts with **exact** training sentences (word-for-word)
- Model can't generalize to new prompts in the same style
- Perplexity drops close to its theoretical floor of **1.0** on training data (the model is nearly
  certain about every next token because it has seen this exact text before) but stays high on
  validation data it hasn't memorized

---

**Pitfall #3: Wrong Max Length**

**Bad:** `max_length=512` on a corpus of short sentences → 90% of every batch is padding  
**Good:** Match `max_length` to your typical paragraph length (128-256 for novels)

**Why it matters:** Wasted computation on padding, slower training, less effective learning

---

**Pitfall #4: No Tokenizer Padding Token**

**Bad:** Forget to set `tokenizer.pad_token` → crash or silent errors  
**Good:** Always set `tokenizer.pad_token = tokenizer.eos_token` for GPT-family models

---

**Quick Health Check After Training:**

```python
# Test 1: Domain knowledge (should work)
generate(model, "Aria Voss checked the Meridian's Promise and")

# Test 2: General knowledge (should still work!)
generate(model, "The capital of France is")

# Test 3: Novel generalization (should work, not memorize)
generate(model, "In the Under-Hold, the rebels gathered and")
```

If test 2 fails → you overtrained (catastrophic forgetting).  
If test 3 is word-for-word from training → shallow memorization.


## Concept 2 (Data-Based): Instructional (Supervised) Fine-Tuning

**Riverside's question for this section:** the model now knows the lore -- but can a ghostwriter
_ask_ it for a continuation, or does it just ramble? An assistant nobody can direct isn't an
assistant.

### The Problem with Continued Pretraining Alone

After continued pretraining, the model knows your domain vocabulary and can continue text in your
style. But try asking it a question:

**You:** `"What are the five tides in the Tidebound world?"`  
**Model (after continued pretraining):** `"What are the five tides in the Tidebound world? This 
question has puzzled scholars for centuries. Some believe there are actually six tides, while..."`
(continues rambling)

**The problem:** The model learned to _continue_ prose, not to _answer questions_ or _follow
instructions_. It will keep generating narrative-style text forever because that's what it was trained
on.

### The Solution: Instruction Tuning (Supervised Fine-Tuning / SFT)

**What it is:** Train on `(prompt, completion)` pairs where the **prompt** is an instruction/question
and the **completion** is the desired response. Crucially, we **mask the prompt tokens** in the loss
so the model is only penalized for the completion portion.

**Key insight:** This teaches the model _behavior_ -- "when you see input shaped like X, respond like
Y" -- rather than just "keep talking like this corpus."

Real-world instruction datasets include:

- [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned) - 52K instruction-following examples
- [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca) - 4.2M GPT-4 completions
- [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1) - 161K human-rated conversations

Here we auto-derive a tiny instruction dataset from Riverside's catalog:

- **Prompt:** `"Continue the fiction narrative in the same style: <paragraph N>"`
- **Completion:** `<paragraph N+1>`

**Pros:**

- Model learns to _follow a format/instruction_, not just continue prose
- Directly usable for chat/assistant interfaces
- Loss masking means the model isn't penalized for "predicting" the prompt it didn't generate

**Cons:**

- Needs actual (prompt, completion) pairs (expensive to create by hand)
- Can narrow diversity toward the exact template it was trained on
- Doesn't fix preference issues (model might follow instructions but in an unhelpful way)

This cell introduces **LoRA** (parameter-efficient tuning) to keep training fast on CPU -- foreshadowing
the budget conversation Riverside's IT lead is going to have with us later.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

INSTRUCTION_PREFIX = "Continue the fiction narrative in the same style:\n\n"


def build_instruction_pairs(novels=None, max_chapters=3):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "cyberpunk", "literary"]  # 5-genre default for richer style diversity

    pairs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        for path in sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]:
            paras = [
                p.strip().replace("\n", " ")
                for p in path.read_text(encoding="utf-8").split("\n\n")
                if len(p.strip()) > 200
            ]
            for a, b in zip(paras, paras[1:]):
                pairs.append(
                    {"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b}
                )
    return pairs


def tokenize_instruction(example, max_length=160, prompt_max_length=96):
    prompt_ids = tokenizer(
        example["prompt"], truncation=True, max_length=prompt_max_length
    )["input_ids"]
    full_text = example["prompt"] + example["completion"]
    tokens = tokenizer(
        full_text, truncation=True, padding="max_length", max_length=max_length
    )
    labels = tokens["input_ids"].copy()
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100  # don't compute loss on the prompt portion
    for i, mask in enumerate(tokens["attention_mask"]):
        if mask == 0:
            labels[i] = -100  # don't compute loss on padding either
    tokens["labels"] = labels
    return tokens


instruction_pairs = build_instruction_pairs(
    novels=["scifi", "fantasy", "mystery", "cyberpunk", "literary"], max_chapters=2
)
print(f"Built {len(instruction_pairs)} instruction pairs from 5 novels")

instruction_dataset = Dataset.from_list(instruction_pairs)
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction, remove_columns=["prompt", "completion"]
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT-2's combined attention projection
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()

training_args_instruct = TrainingArguments(
    output_dir="./checkpoints/instruction-lora",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained("./checkpoints/instruction-lora")
print("Saved instruction-tuned LoRA adapter.")

### Code Walkthrough: Instruction Tuning Cell

**What just ran — four conceptual steps combined into one cell:**

---

**Step A: `build_instruction_pairs()` — creating (prompt, completion) pairs**

```python
pairs.append({"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b})
```

For every pair of consecutive paragraphs `(a, b)` in each chapter, the _preceding_ paragraph becomes the prompt and the _next_ paragraph becomes the completion. This is the cheapest way to auto-generate instruction pairs from raw prose — no human labelling required. The `"\n\n"` delimiter marks where the model should stop echoing the prompt and start generating.

---

**Step B: `tokenize_instruction()` — the prompt-mask pattern**

This is the core difference from continued pretraining's `tokenize_causal()`:

```
Continued pretraining:  [-100 for padding only,  real labels for everything else]
Instruction tuning:     [-100 for prompt + pad,  real labels for completion only]
```

In code:

```python
for i in range(min(len(prompt_ids), len(labels))):
    labels[i] = -100   # mask the entire prompt portion
```

Setting `labels[i] = -100` at prompt positions tells `F.cross_entropy(ignore_index=-100)` to skip those positions when computing the loss. The model is only graded on the _completion_ tokens — never penalised for "predicting" the instruction it was given.

---

**Step C: `LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"])` — LoRA hyperparameters**

| Param                       | Value          | What it controls                                             |
| --------------------------- | -------------- | ------------------------------------------------------------ |
| `r=8`                       | Rank           | Bottleneck dimension — 8 basis vectors to express the update |
| `lora_alpha=16`             | Scaling        | Effective LR multiplier = `alpha/r` = 2.0                    |
| `target_modules=["c_attn"]` | Which layers   | GPT-2 packs Q, K, V into one `c_attn` projection             |
| `lora_dropout=0.05`         | Regularization | Randomly zeros adapter activations during training           |

`get_peft_model(base, config)` wraps every targeted layer with a `LoraLayer` object and freezes all other weights — the only parameters that get optimizer state are the two small adapter matrices per layer.

---

**Step D: `Trainer.train()` — what HuggingFace's Trainer does for you**

Under the hood, one `Trainer.train()` call:

1. Iterates over `train_dataset` in mini-batches of `per_device_train_batch_size=2`
2. Calls `model.forward(input_ids, attention_mask, labels)` → computes cross-entropy loss (ignoring `labels=-100` positions)
3. Calls `loss.backward()` → computes gradients only for `requires_grad=True` parameters (the LoRA matrices)
4. Calls `optimizer.step()` → updates those parameters by `lr × gradient`
5. Logs the loss every `logging_steps=10` steps
6. Stops after `max_steps=60` regardless of dataset size


### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only
**padding** was masked, and every real token was active. Here, the **prompt itself is masked too** --
the model is only ever penalized for generating the completion, never for reproducing the prompt it
was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it
through the real `tokenize_instruction()` used for training, and colors every token position by what
the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)
# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {example_pair['prompt'][:80]!r}...")
print(f"Completion text: {example_pair['completion'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

### Common Pitfalls: Instruction Tuning

**Pitfall #1: Forgetting to Mask the Prompt**

**Bad:** Compute loss on both prompt AND completion → model "predicts" the prompt it was given  
**Good:** Set prompt tokens to `-100` in labels → only penalized for the completion

**Why it matters:**

Without masking:

```python
labels = [3, 822, 25, ...]  # entire sequence including prompt
```

With masking:

```python
labels = [-100, -100, -100, 822, 25, ...]  # first 10 tokens (prompt) masked
```

The model should only learn to **generate the completion**, not memorize the prompt.

---

**Pitfall #2: Template Over-Fitting**

**Bad:** All training pairs use identical prefix: `"Continue the narrative: ..."`  
**Good:** Vary the instruction format, or accept this if you'll always use that prefix at inference

**What happens:** Model becomes "allergic" to prompts without the exact prefix. If you prompt with
just the raw paragraph (no prefix), it won't know what to do.

**Fix:** Either:

1. Use diverse instruction templates during training
2. Always use the exact same prefix at inference (consistency is key)

---

**Pitfall #3: Completion Too Short/Long**

**Bad:** Completions are 5 tokens on average → model learns to be terse  
**Good:** Completions should match your inference expectation (50-100 tokens for narrative)

**Why:** The model learns the **distribution of lengths** from training. If all completions are short,
it will always generate short responses, even when you want more detail.

---

**Pitfall #4: Wrong Learning Rate**

**Bad:** Use the same LR as pretraining (5e-5) for LoRA  
**Good:** LoRA needs **higher LR** (2e-4 to 5e-4) because you're only updating a tiny subset of params

**Rule of thumb:**

- Full fine-tuning: 5e-5 to 1e-4
- Partial freezing: 1e-4 to 2e-4
- LoRA: 2e-4 to 5e-4

Lower rank → higher LR (more aggressive updates needed).

---

**Quick Health Check After Instruction Tuning:**

```python
# Test 1: With the instruction prefix (should work)
prompt = INSTRUCTION_PREFIX + "Aria checked the Meridian and\\n\\n"
generate(instruct_model, prompt)

# Test 2: Without prefix (will likely fail if over-fitted to template)
generate(instruct_model, "Aria checked the Meridian and")

# Test 3: Novel instruction (should generalize)
prompt = INSTRUCTION_PREFIX + "In the Upper decks, Marcus\\n\\n"
generate(instruct_model, prompt)
```

If test 2 produces nonsense → template over-fitting (model expects the prefix).  
If test 3 produces off-topic output → not enough diverse training data.


In [ ]:
# Quick health check after instruction tuning: actually run the three tests described above
print(
    "=== Test 1: With the instruction prefix (should follow the fiction-continuation format) ==="
)
prompt_with_prefix = INSTRUCTION_PREFIX + "Aria checked the Meridian and\n\n"
print(generate(instruct_lora_model, prompt_with_prefix), "\n")

print("=== Test 2: Without the prefix (checks for template over-fitting) ===")
print(generate(instruct_lora_model, "Aria checked the Meridian and"), "\n")

print("=== Test 3: Novel instruction (checks generalization to an unseen prompt) ===")
prompt_novel = INSTRUCTION_PREFIX + "In the Upper decks, Marcus\n\n"
print(generate(instruct_lora_model, prompt_novel))

## Concept 3 (Data-Based): Preference Alignment (RLHF / DPO)

**Riverside's question for this section:** the model follows instructions now -- but does it write
the way Riverside's editors _actually_ like, or just "a" technically-valid continuation? An assistant
that's correct-but-unusable still gets ignored.

### The Problem with Instruction Tuning Alone

After instruction tuning, the model follows instructions. But it might produce outputs that are
_technically correct_ but not what humans actually want:

**An editor asks:** `"Explain quantum entanglement."`  
**Instruction-tuned model:** `"Quantum entanglement is a phenomenon where particles become correlated 
such that the quantum state of one particle cannot be described independently... [continues for 10 
paragraphs with excessive jargon]"`

**Problems:**

- Too verbose (they wanted a 2-sentence explanation)
- Wrong tone (too academic for a casual question)
- Doesn't prioritize what the reader cares about

**The core insight:** Instruction tuning teaches the model to _respond_, but not which responses
humans _prefer_.

### The Solution: Preference Alignment

**The idea:** Show the model pairs of responses to the same prompt -- one that humans prefer
(`chosen`) and one they don't (`rejected`) -- and train it to increase the probability of preferred
responses.

**Two approaches:**

1. **RLHF (Reinforcement Learning from Human Feedback):** Train a separate reward model to score
   responses, then use PPO (reinforcement learning) to optimize the LLM against that reward. (Complex,
   not demoed here.)
2. **DPO (Direct Preference Optimization):** Skip the reward model entirely and optimize directly
   against preference pairs with a closed-form loss. (Simpler, demoed below.)

Real-world preference datasets:

- [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) - 170K human preferences on
  helpfulness/harmlessness
- [ultrafeedback-binarized-preferences-cleaned](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned)
  - 60K preference pairs

**Our stand-in for "what Riverside's editors prefer":**

- **Chosen:** The paragraph that actually follows in the novel (coherent, on-topic continuation --
  the closest thing we have to "an editor approved this")
- **Rejected:** An unrelated paragraph from a different chapter (off-topic, worse continuation)

This is a stand-in for real human preference labels, but demonstrates the mechanics. In a real
Riverside deployment, `chosen`/`rejected` pairs would come from editors actually rating two draft
continuations against each other -- worth keeping in mind for the "how much data would we really need"
question the training run below raises.


### DPO Intuition: What the Loss Is Actually Rewarding

Forget the arithmetic for a second -- here's the whole idea in one sentence: **DPO nudges the model to
like the `chosen` response a bit more, and the `rejected` response a bit less, than a frozen copy of
itself did.**

Walking through it as a story, not a computation:

1. We have two responses to the same prompt: one we like (`chosen`), one we don't (`rejected`).
2. The **policy model** (the one being trained) has some current opinion of how likely each response
   is. So does the **frozen reference model** (a snapshot from before DPO started).
3. For each response, DPO asks: _"Did the policy move its opinion up or down, compared to the
   reference?"_ That movement is the **margin**.
4. DPO then compares the two margins: _"Did the policy move up on `chosen` by more than it moved up
   on `rejected`?"_ If yes, good -- the model is learning the right preference. If no, the loss will
   push back.
5. That comparison gets squashed through a sigmoid (so it behaves like a probability) and turned into
   a loss with a `-log`, exactly like ordinary binary classification: "which response wins?"

The reference model matters because it's an **anchor**: without it, nothing would stop the policy from
just assigning every response a probability of 1 and calling it a day (mode collapse). By measuring
_relative_ movement instead of absolute probability, DPO can only reward genuine preference for
`chosen` over `rejected`, not "be confident about everything."

**Pros:** no separate reward model (unlike PPO-based RLHF), works well with LoRA, directly optimizes
for human preference.

**Cons:** needs paired preference data, can over-optimize ("reward hacking") if beta is too high or
data is noisy, needs a frozen reference model in memory.

<details>
<summary><strong>Optional: the closed-form loss</strong> (skip this if the five steps above are enough -- nothing below adds a new idea, it just names the pieces mathematically)</summary>

$$
\mathcal{L}_{DPO} = -\log \sigma\Big(\beta\big[(\log \pi_\theta(y_w \mid x) - \log \pi_{ref}(y_w
\mid x)) - (\log \pi_\theta(y_l \mid x) - \log \pi_{ref}(y_l \mid x))\big]\Big)
$$

$\pi_\theta$ = current policy, $\pi_{ref}$ = frozen reference, $y_w$ = chosen, $y_l$ = rejected,
$\sigma$ = sigmoid, $\beta$ = temperature. This is exactly steps 3-5 above written symbolically, and
it's the same loss `trl.DPOTrainer` implements -- useful if you're reading the paper or production
code side by side with this notebook, not required to understand what the training loop below does.

</details>

The training loop below is a **simplified, from-scratch implementation** so you can see the mechanics;
it also records the real margin/loss at every step so the chart right after it reflects what actually
happened during training, not a toy example. Production code would use `trl.DPOTrainer`.


In [ ]:
import copy
import torch.nn.functional as F


def build_preference_pairs(novels=None, max_chapters=4, max_pairs=30):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "horror", "literary"]  # 5-genre default

    chapter_files = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        chapter_files.extend(sorted(novel_path.glob("chapter-*.txt"))[:max_chapters])

    all_paragraphs = []
    for path in chapter_files:
        paras = [
            p.strip().replace("\n", " ")
            for p in path.read_text(encoding="utf-8").split("\n\n")
            if len(p.strip()) > 200
        ]
        all_paragraphs.append(paras)

    pairs = []
    for c_idx, paras in enumerate(all_paragraphs):
        other_chapter = all_paragraphs[(c_idx + 1) % len(all_paragraphs)]
        for i in range(len(paras) - 1):
            prompt = f"{INSTRUCTION_PREFIX}{paras[i]}\n\n"
            chosen = paras[i + 1]  # the real, on-topic continuation
            rejected = other_chapter[
                i % len(other_chapter)
            ]  # an unrelated paragraph -> a worse continuation
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pairs[:max_pairs]


def encode_response(prompt, response, max_length=160, prompt_max_length=96):
    """Tokenize prompt+response and return a response_mask marking only the response
    tokens (excluding the shared prompt and any padding) as targets for logprob math."""
    prompt_ids = tokenizer(prompt, truncation=True, max_length=prompt_max_length)[
        "input_ids"
    ]
    full = tokenizer(
        prompt + response, truncation=True, padding="max_length", max_length=max_length
    )
    response_mask = [0] * max_length
    start = min(len(prompt_ids), max_length)
    end = min(sum(full["attention_mask"]), max_length)
    for i in range(start, end):
        response_mask[i] = 1
    return {
        "input_ids": torch.tensor(full["input_ids"]).unsqueeze(0).to(device),
        "attention_mask": torch.tensor(full["attention_mask"]).unsqueeze(0).to(device),
        "response_mask": torch.tensor(response_mask).unsqueeze(0).to(device),
    }


def sequence_logprob(model, input_ids, attention_mask, response_mask):
    """Sum of log P(token_t | tokens<t) over the response-mask positions only."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    targets = input_ids[:, 1:]
    mask = response_mask[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = torch.gather(log_probs, 2, targets.unsqueeze(-1)).squeeze(-1)
    return (token_logprobs * mask).sum(dim=-1)


BETA = 0.1
policy_model = instruct_lora_model  # continue tuning the instruction-tuned LoRA adapter
reference_model = copy.deepcopy(policy_model).eval()
for p in reference_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    [p for p in policy_model.parameters() if p.requires_grad], lr=1e-5
)

preference_pairs = build_preference_pairs(
    novels=["scifi", "fantasy", "mystery", "horror", "literary"], max_chapters=4, max_pairs=30
)
print(f"Built {len(preference_pairs)} preference pairs from 5 novels for the DPO demo")

# Recorded so the visualization right after this cell plots what ACTUALLY happened here,
# instead of a fabricated/illustrative curve.
dpo_history = {
    "step": [],
    "loss": [],
    "chosen_margin": [],
    "rejected_margin": [],
    "preference_diff": [],
}

policy_model.train()
for step, pair in enumerate(preference_pairs):
    chosen = encode_response(pair["prompt"], pair["chosen"])
    rejected = encode_response(pair["prompt"], pair["rejected"])

    policy_chosen_lp = sequence_logprob(policy_model, **chosen)
    policy_rejected_lp = sequence_logprob(policy_model, **rejected)
    with torch.no_grad():
        ref_chosen_lp = sequence_logprob(reference_model, **chosen)
        ref_rejected_lp = sequence_logprob(reference_model, **rejected)

    chosen_margin = policy_chosen_lp - ref_chosen_lp
    rejected_margin = policy_rejected_lp - ref_rejected_lp
    logits = BETA * (chosen_margin - rejected_margin)
    loss = -F.logsigmoid(logits).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    dpo_history["step"].append(step)
    dpo_history["loss"].append(loss.item())
    dpo_history["chosen_margin"].append(chosen_margin.mean().item())
    dpo_history["rejected_margin"].append(rejected_margin.mean().item())
    dpo_history["preference_diff"].append(
        (chosen_margin - rejected_margin).mean().item()
    )

    if step % 5 == 0:
        print(f"step {step:02d} | dpo_loss={loss.item():.4f}")

policy_model.save_pretrained("./checkpoints/preference-dpo")
print("Saved DPO-aligned adapter.")

### Code Walkthrough: DPO Training Loop

**What just ran — the core DPO math implemented in plain PyTorch:**

---

**Step A: `encode_response()` — response mask**

```python
response_mask[i] = 1   # only mark completion tokens as targets
```

Unlike the `Trainer`-based cells, the DPO loop computes log-probabilities manually. `encode_response` tokenizes `prompt + response` together, then builds a `response_mask` that is `1` only over the completion portion. This allows `sequence_logprob` to sum `log P(token | context)` only over the _response_ — not the prompt, not the padding.

---

**Step B: `sequence_logprob()` — how to score a full response**

```python
log_probs = F.log_softmax(logits, dim=-1)           # (batch, seq_len, vocab)
token_logprobs = torch.gather(log_probs, 2, targets.unsqueeze(-1)).squeeze(-1)  # pick the actual next-token's log prob
return (token_logprobs * mask).sum(dim=-1)          # sum over response positions only
```

- `F.log_softmax(logits, dim=-1)` → log-probabilities over the full 50K vocab
- `torch.gather(..., targets)` → extracts only the log-prob of the _ground-truth_ next token at each position
- `* mask` → zeros out prompt positions; `.sum()` → total sequence log-likelihood

---

**Step C: The DPO loss formula**

```python
chosen_margin   = policy_chosen_lp  - ref_chosen_lp    # policy drift for preferred response
rejected_margin = policy_rejected_lp - ref_rejected_lp  # policy drift for dispreferred response
logits_dpo = BETA * (chosen_margin - rejected_margin)
loss = -F.logsigmoid(logits_dpo).mean()
```

- `chosen_margin - rejected_margin > 0` → the policy has shifted _more_ toward chosen than rejected (desired)
- `BETA` controls how strongly the preference margin is enforced (typical: 0.1–0.3)
- `-log sigmoid(...)` is the DPO objective from [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290) — binary cross-entropy over a preference logit rather than token-level loss

---

**Step D: Why `reference_model` must be frozen**

```python
reference_model = copy.deepcopy(policy_model).eval()
for p in reference_model.parameters():
    p.requires_grad = False
```

DPO measures how far the _policy_ has drifted from a stable anchor. If the reference also changed during training, the margins `(policy_lp - ref_lp)` would be meaningless — both models could drift in the same direction and the loss would never converge. `copy.deepcopy` makes a full independent snapshot; setting `requires_grad=False` ensures the reference accumulates _no_ gradients and its weights never move.

---

**Step E: Optimizer scope — only LoRA matrices get updated**

```python
optimizer = torch.optim.AdamW(
    [p for p in policy_model.parameters() if p.requires_grad], lr=1e-5
)
```

Because `policy_model` is the instruction-tuned LoRA adapter, only the adapter matrices have `requires_grad=True`. The frozen GPT-2 base and the reference model never accumulate gradients — optimizer state stays tiny, just like during instruction tuning.


In [ ]:
# Visualize the DPO training dynamics we just recorded above -- real numbers, not a toy example,
# so this chart actually reflects the training that happened in the previous cell.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
steps = np.array(dpo_history["step"])

axes[0].plot(
    steps,
    dpo_history["chosen_margin"],
    "g-o",
    label="chosen margin vs. ref",
    linewidth=2,
    markersize=4,
)
axes[0].plot(
    steps,
    dpo_history["rejected_margin"],
    "r-o",
    label="rejected margin vs. ref",
    linewidth=2,
    markersize=4,
)
axes[0].axhline(0, color="black", linestyle="--", alpha=0.4)
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("Log-prob margin vs. reference")
axes[0].set_title("Policy Drift From Reference")
axes[0].legend(fontsize=8, loc="best")
axes[0].grid(alpha=0.3)

preference_diff = np.array(dpo_history["preference_diff"])
axes[1].plot(steps, preference_diff, "b-o", linewidth=2, markersize=4)
axes[1].axhline(0, color="black", linestyle="--", alpha=0.4, label="No preference")
axes[1].fill_between(
    steps,
    0,
    preference_diff,
    where=preference_diff > 0,
    alpha=0.3,
    color="green",
    label="Prefers chosen",
)
axes[1].fill_between(
    steps,
    0,
    preference_diff,
    where=preference_diff < 0,
    alpha=0.3,
    color="red",
    label="Prefers rejected",
)
axes[1].set_xlabel("Training Step")
axes[1].set_ylabel("Preference difference")
axes[1].set_title("Preference Margin Over Training")
axes[1].legend(fontsize=8, loc="best")
axes[1].grid(alpha=0.3)

axes[2].plot(steps, dpo_history["loss"], "m-o", linewidth=2, markersize=4)
axes[2].set_xlabel("Training Step")
axes[2].set_ylabel("DPO Loss")
axes[2].set_title(
    f"Loss: {dpo_history['loss'][0]:.3f} \u2192 {dpo_history['loss'][-1]:.3f}"
)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("DPO Training Summary (real numbers from the run above):")
print(f"  Initial preference margin: {dpo_history['preference_diff'][0]:.3f}")
print(f"  Final preference margin:   {dpo_history['preference_diff'][-1]:.3f}")
print(f"  Loss: {dpo_history['loss'][0]:.4f} -> {dpo_history['loss'][-1]:.4f}")

# Honest reaction to whatever actually happened above -- not an assumed success story.
# Riverside's stakes: this ran on 30 synthetic preference pairs. One afternoon of two editors
# rating draft continuations against each other wouldn't produce many more than that in real life.
margin_improved = dpo_history["preference_diff"][-1] > dpo_history["preference_diff"][0]
loss_improved = dpo_history["loss"][-1] < dpo_history["loss"][0]
print(f"\n{'=' * 80}")
if margin_improved and loss_improved:
    print(
        "Reading this run: preference margin AND loss both moved in the right direction -- "
        "on this run's numbers, 30 pairs was (barely) enough signal."
    )
else:
    print(
        "Reading this run HONESTLY: the numbers above did not cleanly improve "
        f"({'margin regressed' if not margin_improved else 'margin OK'}, "
        f"{'loss went up' if not loss_improved else 'loss OK'}). That's a real, useful result for "
        "Riverside, not a notebook bug: with only 30 preference pairs and one pass through them, "
        "there isn't enough signal for DPO to reliably converge. Before shipping preference "
        "alignment to production, Riverside would need either (a) meaningfully more preference "
        "pairs (hundreds to thousands, not 30), (b) multiple epochs over the pairs it has, or "
        "(c) a smaller learning rate with more steps -- not just 'run DPO and ship it.'"
    )
print(f"{'=' * 80}")

### Common Pitfalls: DPO (Preference Alignment)

**Pitfall #1: Weak or Noisy Preference Pairs**

**Bad:** `chosen` and `rejected` are nearly identical or have inconsistent quality  
**Good:** Clear preference signal — chosen should be noticeably better than rejected

**Example of BAD pair:**

- Chosen: `"Quantum entanglement means particles are connected."`
- Rejected: `"Quantum entanglement means particles are linked."` ← too similar!

**Example of GOOD pair:**

- Chosen: `"Quantum entanglement means particles are connected."`
- Rejected: `"This is a complicated quantum mechanical phenomenon involving wave function collapse and non-local correlations that [500 more words of jargon]"` ← clearly worse!

**Why it matters:** If the preference signal is weak, the model won't learn what humans actually want.

---

**Pitfall #2: Beta (β) Tuned Incorrectly**

**Bad:** β = 1.0 → over-aggressive preference signal, mode collapse  
**Bad:** β = 0.01 → too weak, model barely changes  
**Good:** Start with β = 0.1 to 0.3 and tune based on validation

**What β does:**

- **High β (0.5-1.0):** Strong preference signal → fast learning, but risk of "reward hacking" (model
  exploits the preference distribution)
- **Low β (0.01-0.05):** Weak signal → slow learning, safer
- **Medium β (0.1-0.3):** Balanced (most common in practice)

**Symptom of bad β:**

- β too high → model generates identical text for all prompts (mode collapse)
- β too low → model ignores preferences entirely, no improvement

---

**Pitfall #3: Forgetting to Freeze the Reference Model**

**Bad:** Reference model continues training → DPO loss becomes meaningless  
**Good:** `reference_model.eval()` and `param.requires_grad = False` for all ref params

**Why:** The reference model is the "anchor" — it must stay fixed to measure how far the policy has
drifted. If it changes, the loss calculation breaks.

---

**Pitfall #4: Training on Top of Base Model Instead of Instruction-Tuned**

**Bad:** Run DPO on a raw pretrained model  
**Good:** DPO should be the **final stage** after instruction tuning

**Why:** DPO assumes the model already knows how to follow instructions. If you DPO a base model, it
will learn preferences over gibberish continuations, not helpful responses.

**Correct pipeline:**

```
Base model → Continued pretraining → Instruction tuning → DPO
```

---

**Pitfall #5: Not Monitoring Divergence from Reference**

**Bad:** Train DPO for 1000 steps without checking KL divergence  
**Good:** Monitor `KL(policy || reference)` — stop if it exceeds 1.0-2.0

**Why:** DPO can cause the policy to drift too far from the reference, leading to nonsensical but
"high-scoring" outputs (reward hacking).

---

**Quick Health Check After DPO:**

```python
# Test 1: Preferred style (should be concise)
generate(dpo_model, INSTRUCTION_PREFIX + "Explain quantum entanglement.\\n\\n")

# Test 2: Still coherent on domain tasks
generate(dpo_model, INSTRUCTION_PREFIX + "Continue: Aria checked the panel and\\n\\n")

# Test 3: Doesn't mode collapse (vary prompts, should get varied outputs)
for i in range(3):
    generate(dpo_model, INSTRUCTION_PREFIX + f"Describe the Meridian (attempt {i}).\\n\\n")
```

If test 1 is still verbose → β too low or not enough training.  
If test 3 produces identical output 3 times → mode collapse (β too high).


In [ ]:
# Quick health check after DPO: actually run the three tests described above
print(
    "=== Test 1: Preferred style (should be more concise than the pre-DPO instruction model) ==="
)
print(
    generate(policy_model, INSTRUCTION_PREFIX + "Explain quantum entanglement.\n\n"),
    "\n",
)

print("=== Test 2: Still coherent on domain tasks ===")
print(
    generate(
        policy_model, INSTRUCTION_PREFIX + "Continue: Aria checked the panel and\n\n"
    ),
    "\n",
)

print("=== Test 3: Doesn't mode collapse (vary prompts, should get varied outputs) ===")
for i in range(3):
    print(f"--- attempt {i} ---")
    print(
        generate(
            policy_model,
            INSTRUCTION_PREFIX + f"Describe the Meridian (attempt {i}).\n\n",
        )
    )

## Parameter-Based Axis: How Many Weights Do We Actually Update?

**Riverside's question for this section:** IT has given us one laptop CPU and a deadline. We now have
a model that knows the lore, follows instructions, and matches editorial taste -- but which of the
three ways to _train_ it can we actually afford to run and re-run as the catalog grows?

### The Cost Problem

All three data-based techniques (continued pretraining, instruction tuning, preference alignment) work
by gradient descent on model weights. But **updating all weights is expensive:**

- **Memory:** ~355M parameters × (4 bytes per param + 8 bytes optimizer state) = ~4.3 GB just for
  `gpt2-medium`. For 70B models, this becomes **840 GB**.
- **Compute:** More trainable params = longer training time
- **Risk:** Full updates can "overwrite" the model's general knowledge (catastrophic forgetting) --
  which for Riverside means the assistant could start forgetting ordinary English while it's busy
  memorizing the sci-fi glossary.

### The Trade-Off Spectrum

Independent of _what data_ you train on, you can choose _how much of the model_ to update:

| Technique            | Trainable % | Memory  | Quality | Forgetting Risk | When to Use                         |
| -------------------- | ----------- | ------- | ------- | --------------- | ----------------------------------- |
| **Full fine-tuning** | 100%        | Highest | Highest | Highest         | Small models, abundant compute      |
| **Partial freezing** | 10-30%      | Medium  | Medium  | Medium          | Limited budget, want more than PEFT |
| **LoRA**             | <1%         | Lowest  | High    | Lowest          | Most production scenarios today     |

The following cells apply all three strategies to the _same_ continued-pretraining objective, so
parameter counts are directly comparable -- this is the data Riverside's IT lead will actually want to
see before signing off on a training budget.

---

### Concept 4 (Parameter-Based): Full Fine-Tuning

**What it is:** Every single weight in the model is unfrozen and updated by the optimizer.

**Pros:** The model has maximum "room" to adapt to Riverside's catalog.

**Cons:**

- Costs the most memory/compute
- Highest risk of catastrophic forgetting if the corpus is small (Riverside's 7 novels are tiny by
  LLM standards)
- For large models (>7B params), often infeasible without multi-GPU setups -- a non-starter for a
  publisher with one laptop

We already ran this in Concept 1 (`./checkpoints/non-instruction-full`) -- that cell **is** full
fine-tuning. The cell below quantifies what "100% trainable" looks like for `gpt2-medium`.


In [ ]:
param_check_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total = sum(p.numel() for p in param_check_model.parameters())
trainable = sum(p.numel() for p in param_check_model.parameters() if p.requires_grad)
print(
    f"Full fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.1f}%)"
)
del param_check_model

### Concept 5 (Parameter-Based): Partial Fine-Tuning (Layer Freezing)

**Riverside's question for this section:** if full fine-tuning is the "maximum quality, maximum
laptop-fan-noise" option, is there a middle ground that still lets us re-train every time the catalog
grows, without waiting hours?

**The observation:** In transformer models, **early layers** learn general language features
(tokenization, basic syntax, common words) while **later layers** learn task-specific patterns. This
is similar to how early layers in CNNs detect edges, while later layers detect objects.

**The strategy:** Freeze everything, then selectively unfreeze:

- The last N transformer blocks (task-specific adaptation)
- The output head (final projection to vocabulary)

**Pros:**

- Much cheaper than full fine-tuning (only 10-30% of parameters)
- Less prone to catastrophic forgetting (general features preserved)
- No new architecture needed

**Cons:**

- Still edits raw model weights (can't easily "swap" like an adapter) -- so Riverside couldn't keep
  one editing-assistant checkpoint and one knowledge-base checkpoint without storing two full copies
  of `gpt2-medium`
- Choosing _how many_ layers to unfreeze is a manual hyperparameter
- Middle ground: not as cheap as LoRA, not as powerful as full fine-tuning

**Example:** For `gpt2-medium` (24 transformer blocks), we unfreeze only the last ~25% of blocks
(6 blocks) + output head.


### Visualizing Layer-by-Layer Freezing

Before we run the code, let's visualize **exactly which layers** in `gpt2-medium` (24 transformer
blocks) will be frozen vs. trainable when we unfreeze the last ~25% of blocks.

**The intuition:**

Think of the transformer as a **semantic refinement pipeline**:

| Layer                         | What it learns                                    | Freeze or Train? | Why?                                             |
| ----------------------------- | ------------------------------------------------- | ---------------- | ------------------------------------------------ |
| **Early blocks (~first 50%)** | Basic syntax, common words, tokenization patterns | FROZEN           | These are universal — no need to change          |
| **Middle blocks (~50-75%)**   | Mid-level semantics, phrase structure             | FROZEN           | Still mostly general-purpose                     |
| **Last ~25% of blocks**       | Task-specific patterns, domain adaptation         | TRAINABLE        | This is where domain/task specialization happens |
| **Output head**               | Final vocabulary distribution                     | TRAINABLE        | Must learn domain-specific words                 |

**Why this works:**

1. **Early layers = general features:** Just like CNNs learn edges in early layers, transformer early
   blocks learn general language structure that's useful for _any_ task.
2. **Late layers = task-specific:** The final blocks learn task/domain-specific patterns. By only
   training these, we adapt to our corpus without forgetting general English.
3. **Catastrophic forgetting prevention:** Freezing ~75% of the model preserves general language
   ability while allowing focused adaptation.


In [ ]:
# Visualize partial freezing: which layers are trainable?
from transformers import AutoConfig
from matplotlib.patches import Patch

_freeze_cfg = AutoConfig.from_pretrained(MODEL_NAME)
n_layers = _freeze_cfg.n_layer  # gpt2-medium has 24 transformer blocks
unfreeze_from = n_layers - max(2, n_layers // 4)  # unfreeze the last ~25% of blocks
layers = [f"Block {i}" for i in range(n_layers)] + ["Output Head"]
layer_positions = np.arange(len(layers))
tick_stride = max(1, n_layers // 12)  # keep y-axis labels readable regardless of depth

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, n_layers * 0.3)))

# Plot 1: Frozen vs Trainable blocks. With 24+ rows there isn't room for per-bar text labels
# without them overlapping, so color + a legend carries the FROZEN/TRAINABLE distinction instead.
colors = ["lightblue" if i < unfreeze_from else "coral" for i in range(n_layers)] + [
    "coral"
]
ax1.barh(
    layer_positions, [1] * len(layers), color=colors, edgecolor="black", linewidth=1.0
)

ax1.set_yticks(layer_positions[::tick_stride])
ax1.set_yticklabels([layers[i] for i in layer_positions[::tick_stride]])
ax1.set_xlim(0, 1)
ax1.set_xticks([])
ax1.set_title(
    f"Partial Fine-Tuning Strategy\n({MODEL_NAME}: {n_layers} blocks)",
    fontsize=12,
    fontweight="bold",
)
ax1.invert_yaxis()
ax1.legend(
    handles=[
        Patch(facecolor="lightblue", edgecolor="black", label="FROZEN"),
        Patch(facecolor="coral", edgecolor="black", label="TRAINABLE"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    fontsize=9,
)

# Plot 2: Gradient flow visualization
gradient_flow = (
    [0.0] * unfreeze_from
    + list(np.linspace(0.5, 1.0, n_layers - unfreeze_from))
    + [1.0]
)
ax2.barh(
    layer_positions,
    gradient_flow,
    color="green",
    alpha=0.7,
    edgecolor="black",
    linewidth=1.0,
)
ax2.set_yticks(layer_positions[::tick_stride])
ax2.set_yticklabels([layers[i] for i in layer_positions[::tick_stride]])
ax2.set_xlabel("Gradient Magnitude (relative)", fontsize=10)
ax2.set_title("Gradient Flow During Backpropagation", fontsize=12, fontweight="bold")
ax2.invert_yaxis()
ax2.set_xlim(0, 1.1)

frozen_mid = unfreeze_from / 2
trainable_mid = unfreeze_from + (n_layers - unfreeze_from) / 2
ax2.text(
    0.05,
    frozen_mid,
    "No gradient\nflow",
    ha="center",
    va="center",
    fontsize=9,
    color="gray",
    fontweight="bold",
    style="italic",
)
ax2.text(
    0.75,
    trainable_mid,
    "Full gradient\nflow",
    ha="center",
    va="center",
    fontsize=9,
    color="darkgreen",
    fontweight="bold",
)

plt.tight_layout()
plt.show()

# Calculate parameter breakdown
total_blocks = n_layers
frozen_blocks = unfreeze_from
trainable_blocks = total_blocks - frozen_blocks
frozen_pct = (frozen_blocks / total_blocks) * 100
trainable_pct = 100 - frozen_pct

print(f"\n{'=' * 70}")
print(f"Partial Fine-Tuning Configuration:")
print(f"{'=' * 70}")
print(f"  Total blocks:      {total_blocks}")
print(
    f"  Frozen blocks:     {frozen_blocks} (blocks 0-{frozen_blocks-1}) — {frozen_pct:.1f}%"
)
print(
    f"  Trainable blocks:  {trainable_blocks} (blocks {unfreeze_from}-{total_blocks-1}) — {trainable_pct:.1f}%"
)
print(f"  Output head:       TRAINABLE")
print(f"{'=' * 70}")
print(f"  Memory savings:    ~{frozen_pct:.0f}% less optimizer state")
print(f"  Forgetting risk:   LOW (general features preserved)")
print(f"  Adaptation power:  MEDIUM (targeted domain learning)")
print(f"{'=' * 70}")

In [ ]:
freeze_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

for param in freeze_model.parameters():
    param.requires_grad = False

n_layers = freeze_model.config.n_layer  # gpt2-medium has 24 transformer blocks
unfreeze_from = n_layers - max(
    2, n_layers // 4
)  # unfreeze only the last ~25% of blocks

for name, param in freeze_model.named_parameters():
    if any(f"h.{i}." in name for i in range(unfreeze_from, n_layers)):
        param.requires_grad = True
    if "ln_f" in name or "lm_head" in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in freeze_model.parameters())
print(
    f"Partial fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.2f}%)"
)

freeze_dataset = Dataset.from_dict(
    {"text": load_corpus_paragraphs(novels=["fantasy", "cyberpunk"], max_chapters=3)}
)
freeze_tokenized = freeze_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

training_args_freeze = TrainingArguments(
    output_dir="./checkpoints/partial-freeze",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=1e-4,
    report_to="none",
)

trainer_freeze = Trainer(
    model=freeze_model, args=training_args_freeze, train_dataset=freeze_tokenized
)
trainer_freeze.train()
freeze_model.save_pretrained("./checkpoints/partial-freeze")
print("Saved partial (layer-freezing) fine-tune checkpoint.")

### Cracking Open Partial Freezing: Did the Frozen Blocks Actually Stay Frozen?

This is the claim partial freezing makes: only the last ~25% of blocks should move, and everything
before that should be bit-for-bit unchanged. Let's not take that on faith -- measure the real
per-block weight delta the same way we just did for full fine-tuning, using the same untouched
`base_model` as the reference.


In [ ]:
# Real per-block weight-delta norms: freeze_model (after training) vs. base_model (never trained).
# Colors mark which blocks were UNFROZEN during training, so we can check the claim visually.
freeze_block_deltas = []
base_state = dict(base_model.named_parameters())
for i in range(base_model.config.n_layer):
    delta_norm_sq = 0.0
    for name, p in freeze_model.named_parameters():
        if f"h.{i}." in name:
            delta = p.data - base_state[name].data
            delta_norm_sq += delta.norm().item() ** 2
    freeze_block_deltas.append(delta_norm_sq**0.5)

freeze_bar_colors = [
    "lightblue" if i < unfreeze_from else "coral"
    for i in range(base_model.config.n_layer)
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    range(len(freeze_block_deltas)),
    freeze_block_deltas,
    color=freeze_bar_colors,
    alpha=0.9,
)
ax.set_xlabel("Transformer block")
ax.set_ylabel("||W_after - W_before|| (real)")
ax.set_title(
    "Partial Freezing: Real Per-Block Weight Movement (blue=frozen, coral=trainable)",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

frozen_deltas = freeze_block_deltas[:unfreeze_from]
trainable_deltas = freeze_block_deltas[unfreeze_from:]
print(
    f"Frozen blocks (0-{unfreeze_from - 1}): max real delta = {max(frozen_deltas):.2e} "
    f"(should be ~0 -- these were never touched)"
)
print(
    f"Trainable blocks ({unfreeze_from}-{base_model.config.n_layer - 1}): "
    f"delta range = [{min(trainable_deltas):.3f}, {max(trainable_deltas):.3f}] "
    f"(these are where all the learning happened)"
)
print(
    "\nThis is the real proof behind partial freezing's pitch to Riverside: the config claims "
    f"{unfreeze_from}/{base_model.config.n_layer} blocks are frozen, and the measured weights "
    "confirm it exactly -- not just fewer optimizer steps, but literally zero movement in the "
    "blocks we said wouldn't move."
)

### Concept 6 (Parameter-Based): Parameter-Efficient Fine-Tuning (LoRA)

**Riverside's question for this section:** remember the two jobs -- an editing assistant _and_ a
knowledge base? With full fine-tuning or partial freezing, shipping both means storing two full
355M-parameter models. Is there a way to keep one frozen base model on disk and swap in a small,
task-specific adapter depending on who's asking?

**The key insight:** Instead of updating existing weights, **freeze the entire base model** and inject
small trainable matrices alongside key weight matrices.

**How LoRA works:**

For a weight matrix $W$ (e.g., attention projection), instead of updating $W \rightarrow W + \Delta W$,
we:

1. **Freeze** $W$ (no updates ever)
2. **Add** a low-rank decomposition: $\Delta W = BA$, computed as $B(Ax)$ -- $x$ hits $A$ first:
   - $A$ is $r \times d$ (rank-**reducing** projection: takes the $d$-dim input down to $r$ dims)
   - $B$ is $d \times r$ (rank-**expanding** projection: takes that $r$-dim result back up to $d$ dims)
   - $r \ll d$ (rank is much smaller than original dimension)

**Example:** For `gpt2-medium`'s attention (d=1024), with r=8:

- Original: 1024 × 1024 = **1,048,576 parameters**
- LoRA: (8×1024) + (1024×8) = **16,384 parameters** (1.56% of original)

**What to tune:**

- `r` (rank): Higher = more capacity but more parameters. Typical: 4-64.
- `target_modules`: Which weight matrices to adapt. For transformers: attention projections (`q`,
  `k`, `v`, `o`) and sometimes feed-forward layers.
- `lora_alpha`: Scaling factor (typical: 2×r)

**Pros:**

- **Tiny memory footprint:** Only adapter's optimizer state needed
- **Swappable:** Train multiple adapters on the same frozen base model, swap at inference -- this is
  the direct answer to Riverside's two-jobs problem: one frozen `gpt2-medium` on disk, an
  "editing-assistant" adapter and a "knowledge-base" adapter (a few MB each) swapped in per request
- **Mergeable:** Can merge $BA$ into $W$ for zero-latency deployment
- **Lowest forgetting risk:** Base weights never change

**Cons:**

- Slightly lower quality ceiling than full fine-tuning for extreme distribution shifts
- Adds hyperparameters to tune (`r`, `alpha`, `target_modules`)
- Inference needs adapter loaded/merged

**Related techniques not demoed here:**

- **Adapters:** Small bottleneck layers inserted between transformer blocks
- **Prefix tuning:** Learn virtual tokens prepended to input
- **QLoRA:** LoRA on top of 4-bit quantized base model (GPU-specific)

This cell applies LoRA to _continued pretraining_ (not instruction tuning) to show the parameter axis
and data axis are independent choices.


### LoRA Decomposition: A Visual Intuition

Let's visualize exactly **what LoRA does** with concrete matrix dimensions. We'll use a tiny example
to make it crystal clear.

**Scenario:** `gpt2-medium`'s attention weight matrix `W_attn` is 1024×1024 (1,048,576 parameters).

**Full fine-tuning** would update: `W_new = W_old + ΔW` where ΔW is also 1024×1024 (1,048,576
trainable params).

**LoRA** instead uses: `W_new = W_old + B·A` where:

- `B` is 1024×8 (8,192 parameters)
- `A` is 8×1024 (8,192 parameters)
- **Total:** 16,384 trainable parameters (1.56% of the original!)

**Key insight:** The rank bottleneck (r=8) forces the update to live in a low-dimensional subspace.
This is like saying "all the adaptation you need can be expressed as 8 basis vectors" instead of the
full 1024-dimensional freedom.

**Why does this work?**

1. **Task adaptations are low-rank:** Fine-tuning for a specific task doesn't need to change every
   direction in the 1024D space — most of the "general language understanding" can stay frozen.
2. **Overfitting resistance:** With fewer parameters, LoRA is less likely to memorize the training
   data and more likely to learn generalizable patterns.
3. **Efficient gradient flow:** The low-rank bottleneck acts as a regularizer, concentrating gradient
   updates into the most important directions.


In [ ]:
# Visualize LoRA matrix decomposition
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

d, r = base_model.config.n_embd, 8  # dimension (gpt2-medium: 1024) and rank

# Plot 1: Full fine-tuning ΔW
axes[0].add_patch(Rectangle((0, 0), d, d, fill=True, color="steelblue", alpha=0.6))
axes[0].set_xlim(0, d)
axes[0].set_ylim(0, d)
axes[0].set_aspect("equal")
axes[0].set_title(
    f"Full Fine-Tuning: ΔW\n{d}×{d} = {d*d:,} trainable params", fontsize=11
)
axes[0].set_xlabel(f"{d}")
axes[0].set_ylabel(f"{d}")
axes[0].text(
    d / 2,
    d / 2,
    f"{d*d:,}\nparameters",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
)

# Plot 2: LoRA matrix A -- hits the input FIRST, projects d dims DOWN to r dims
axes[1].add_patch(Rectangle((0, 0), d, r, fill=True, color="mediumseagreen", alpha=0.7))
axes[1].set_xlim(0, d)
axes[1].set_ylim(0, r + 100)
axes[1].set_aspect("equal")
axes[1].set_title(
    f"LoRA Matrix A (down-project)\n{r}×{d} = {r*d:,} params", fontsize=11
)
axes[1].set_xlabel(f"{d}")
axes[1].set_ylabel(f"{r}")
axes[1].text(
    d / 2,
    r / 2,
    f"{r*d:,}\nparams",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)

# Plot 3: LoRA matrix B -- takes A's r-dim output and projects it back UP to d dims
axes[2].add_patch(Rectangle((0, 0), r, d, fill=True, color="coral", alpha=0.7))
axes[2].set_xlim(0, r + 100)
axes[2].set_ylim(0, d)
axes[2].set_aspect("equal")
axes[2].set_title(f"LoRA Matrix B (up-project)\n{d}×{r} = {d*r:,} params", fontsize=11)
axes[2].set_xlabel(f"{r}")
axes[2].set_ylabel(f"{d}")
axes[2].text(
    r / 2,
    d / 2,
    f"{d*r:,}\nparams",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)

# Plot 4: B(Ax) result (low-rank approximation)
axes[3].add_patch(Rectangle((0, 0), d, d, fill=True, color="mediumpurple", alpha=0.6))
axes[3].set_xlim(0, d)
axes[3].set_ylim(0, d)
axes[3].set_aspect("equal")
axes[3].set_title(
    f"B(Ax) = Low-Rank ΔW\n{d}×{d} with rank {r}\nTotal: {d*r + r*d:,} params",
    fontsize=11,
)
axes[3].set_xlabel(f"{d}")
axes[3].set_ylabel(f"{d}")
axes[3].text(
    d / 2,
    d / 2,
    f"rank-{r}\nupdate",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
)
axes[3].annotate(
    "",
    xy=(d / 2, d - 50),
    xytext=(d / 2, 50),
    arrowprops=dict(arrowstyle="<->", lw=2, color="yellow"),
)
axes[3].text(
    d / 2 + 80,
    d / 2,
    f"Only {r} degrees\nof freedom!",
    fontsize=9,
    color="yellow",
    fontweight="bold",
)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# Print parameter savings
full_params = d * d
lora_params = d * r + r * d
saving_pct = (1 - lora_params / full_params) * 100

print(f"\n{'=' * 70}")
print(f"Parameter Efficiency Analysis (d={d}, r={r}):")
print(f"{'=' * 70}")
print(f"  Full fine-tuning:  {full_params:>10,} parameters (100.0%)")
print(
    f"  LoRA adaptation:   {lora_params:>10,} parameters ({lora_params/full_params*100:>5.2f}%)"
)
print(
    f"  Savings:           {full_params - lora_params:>10,} parameters ({saving_pct:>5.2f}%)"
)
print(f"{'=' * 70}")
print(f"  Memory savings: ~{saving_pct:.1f}% less optimizer state")
print(f"  Training speed: ~{saving_pct:.1f}% fewer gradients to compute")
print(f"  Swappability: Can load/unload adapters in <1 second")
print(f"{'=' * 70}")

### Cracking Open a Real LoRA-Adapted Model

The diagram above is the toy math (`d x d`, `r=8`, hand-picked numbers). Let's now open
`instruct_lora_model` -- the real LoRA adapter we already trained earlier for instruction tuning --
and look at the actual PyTorch modules PEFT injected, then trace one real forward pass through an
adapted layer to see the frozen base output and the tiny LoRA delta side by side, on real activations
instead of shapes.


In [ ]:
# Find every module PEFT actually injected LoRA adapters into
lora_layers = [
    (name, module)
    for name, module in instruct_lora_model.named_modules()
    if hasattr(module, "lora_A") and len(getattr(module, "lora_A")) > 0
]
print(
    f"PEFT wrapped {len(lora_layers)} layers with LoRA adapters "
    f"(one c_attn per transformer block)."
)
print("First 3 wrapped module names:")
for name, _ in lora_layers[:3]:
    print(" ", name)

# Zoom into the very first block's adapted attention projection
name0, layer0 = lora_layers[0]
base0 = layer0.base_layer
lora_A0 = layer0.lora_A["default"]
lora_B0 = layer0.lora_B["default"]
scaling0 = layer0.scaling["default"]

print(f"\nInside {name0}:")
print(
    f"  Frozen base layer:   {type(base0).__name__}, weight shape {tuple(base0.weight.shape)}"
)
print(f"  lora_A (down-proj):  {tuple(lora_A0.weight.shape)}  <- trainable")
print(f"  lora_B (up-proj):    {tuple(lora_B0.weight.shape)}  <- trainable")
print(f"  scaling (alpha/r):   {scaling0}")
print(
    f"  Trained lora_B norm: {lora_B0.weight.norm().item():.4f}  "
    f"(this was ~0.0 before training -- lora_B is zero-initialized so the "
    f"adapter starts as a no-op)"
)

### Tracing a Real Forward Pass Through the Adapted Layer

`lora_B` started at all-zeros, so before training this adapter was a mathematical no-op: the combined
output was _exactly_ the frozen base output. After training, `lora_B` has real values (norm printed
above), so now it nudges the output. Let's actually capture both paths -- the frozen base output and
the full (base + LoRA) output -- for a real prompt, using forward hooks (no reimplementing the math by
hand, so there's no risk of getting the internal computation subtly wrong).


In [ ]:
# Capture the frozen base output and the combined (base + LoRA) output for a real prompt,
# using forward hooks -- this measures what PEFT actually computes, not a re-derivation of it.
captured = {}


def make_hook(key):
    def hook(module, inputs, output):
        captured[key] = output.detach()

    return hook


hook_base = base0.register_forward_hook(make_hook("base_only"))
hook_combined = layer0.register_forward_hook(make_hook("combined"))

demo_prompt = INSTRUCTION_PREFIX + "Aria Voss checked the Meridian's Promise and"
enc_lora = tokenizer(demo_prompt, return_tensors="pt").to(device)

instruct_lora_model.eval()
with torch.no_grad():
    _ = instruct_lora_model(**enc_lora)

hook_base.remove()
hook_combined.remove()

base_out    = captured["base_only"][0]   # (seq_len, 3072) — frozen W0 @ x + bias
combined_out = captured["combined"][0]   # (seq_len, 3072) — base_out + scaling * B(A(x))
lora_delta  = combined_out - base_out    # isolate just the adapter's real contribution

last_pos = base_out.shape[0] - 1        # last token position carries the richest context
show_dims = 60                           # first 60 of 3072 c_attn dimensions

print(f"Prompt: {demo_prompt!r}")
print(f"c_attn output dim: {base_out.shape[-1]} (= 3 × n_embd, packed Q/K/V for gpt2-medium)")
print(f"\nAt the last token position:")
print(f"  ||base output||   = {base_out[last_pos].norm().item():.3f}")
print(f"  ||LoRA delta||    = {lora_delta[last_pos].norm().item():.5f}")
print(
    f"  delta / base norm = {(lora_delta[last_pos].norm() / base_out[last_pos].norm()).item():.4%}"
    " ← the adapter nudges the output by a small fraction, it doesn't replace it"
)

# ── Static panel: Base vs Combined + LoRA Delta at last position ─────────────
fig_static, (ax_static1, ax_static2) = plt.subplots(1, 2, figsize=(14, 4))

ax_static1.plot(base_out[last_pos, :show_dims].numpy(),
                color="steelblue", label="base (frozen W0 × x)")
ax_static1.plot(combined_out[last_pos, :show_dims].numpy(),
                color="coral", linestyle="--", label="combined (base + LoRA)")
ax_static1.set_title("Base vs. Combined Output\n(first 60 of 3072 c_attn dims)", fontsize=11)
ax_static1.set_xlabel("Output dimension")
ax_static1.legend(fontsize=8)
ax_static1.grid(alpha=0.3)

ax_static2.bar(np.arange(show_dims), lora_delta[last_pos, :show_dims].numpy(),
               color="mediumseagreen")
ax_static2.set_title(
    f"LoRA Delta at token position {last_pos} (the last token)\n"
    "What lora_B(lora_A(x)) × scaling actually adds", fontsize=11)
ax_static2.set_xlabel("Output dimension")
ax_static2.axhline(0, color="black", linewidth=0.8)
ax_static2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Animated panel: watch the LoRA delta evolve token by token ───────────────
# Each animation frame reveals the delta bar-chart for one more token position,
# so you can see how the adapter's correction changes as the model builds context.
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_tokens  = lora_delta.shape[0]
delta_arr = lora_delta[:, :show_dims].numpy()
delta_max = np.abs(delta_arr).max() * 1.15 or 1e-6   # graceful fallback if all zeros

fig_anim, ax_anim = plt.subplots(figsize=(12, 4))
fig_anim.patch.set_facecolor("#f8f8f8")
ax_anim.set_facecolor("#f8f8f8")

bar_colors  = ["mediumseagreen" if v >= 0 else "coral" for v in delta_arr[0]]
bars_anim   = ax_anim.bar(np.arange(show_dims), delta_arr[0], color=bar_colors)
ax_anim.axhline(0, color="black", linewidth=0.8)
ax_anim.set_ylim(-delta_max, delta_max)
ax_anim.set_xlim(-1, show_dims)
ax_anim.set_xlabel("Output dimension (first 60 of 3072)", fontsize=10)
ax_anim.set_ylabel("LoRA delta value", fontsize=10)
ax_anim.grid(alpha=0.3)

# Show the decoded token at each position for context
tokens_decoded = tokenizer.convert_ids_to_tokens(enc_lora["input_ids"][0].tolist())

title_obj = ax_anim.set_title("", fontsize=11, fontweight="bold")

def _update(frame):
    deltas = delta_arr[frame]
    for bar, d in zip(bars_anim, deltas):
        bar.set_height(d)
        bar.set_color("mediumseagreen" if d >= 0 else "coral")
    tok = tokens_decoded[frame] if frame < len(tokens_decoded) else "?"
    title_obj.set_text(
        f"LoRA Delta — token {frame}/{n_tokens - 1}  '{tok}'\n"
        f"||delta|| = {np.linalg.norm(deltas):.5f}"
    )
    return list(bars_anim) + [title_obj]

anim = FuncAnimation(fig_anim, _update, frames=n_tokens, interval=160, blit=False)
plt.close(fig_anim)    # suppress the static inline display; the HTML widget takes over

print(
    f"\nAnimation: {n_tokens} frames, one per token in the prompt.\n"
    "Green bars = positive delta (adapter boosts this c_attn dimension).\n"
    "Red bars   = negative delta (adapter suppresses it).\n"
    "Watch the pattern shift as context accumulates — early tokens show almost no signal,\n"
    "late tokens (the last few) carry the richest context so the adapter fires harder.\n"
)
display(HTML(anim.to_jshtml(fps=6)))

print(
    "\nThis is the whole story: the frozen Conv1D still does all the heavy lifting (base output), "
    "and the tiny rank-8 adapter adds a small, learned correction on top — computed fresh for "
    "every token position, every forward pass."
)

In [ ]:
lora_config_pt = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
)

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = get_peft_model(lora_pt_base, lora_config_pt)
lora_pt_model.print_trainable_parameters()

lora_pt_dataset = Dataset.from_dict(
    {
        "text": load_corpus_paragraphs(
            novels=["mystery", "horror", "literary"], max_chapters=3
        )
    }
)
lora_pt_tokenized = lora_pt_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

training_args_lora_pt = TrainingArguments(
    output_dir="./checkpoints/peft-lora",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

trainer_lora_pt = Trainer(
    model=lora_pt_model, args=training_args_lora_pt, train_dataset=lora_pt_tokenized
)
trainer_lora_pt.train()
lora_pt_model.save_pretrained("./checkpoints/peft-lora")
print("Saved parameter-efficient (LoRA) continued-pretraining adapter.")

### Visual Comparison: Parameter Counts Across All Techniques

Before diving into the LoRA code, let's visualize **exactly how much memory/compute each parameter-
based approach requires**. This builds the intuition for why LoRA has become the industry standard.


In [ ]:
# Visual parameter comparison across all techniques
from matplotlib.patches import FancyBboxPatch

# Real parameter counts pulled from the actual models trained earlier in this notebook
# (not hardcoded, so this stays correct no matter which base model MODEL_NAME points to)
total_params = sum(p.numel() for p in base_model.parameters())
full_ft_params = total_params  # 100%
partial_ft_params = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
lora_params = sum(
    p.numel() for p in instruct_lora_model.parameters() if p.requires_grad
)

techniques = ["Full\nFine-Tuning", "Partial\nFreezing", "LoRA"]
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]
colors = ["steelblue", "coral", "mediumseagreen"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Absolute parameter counts (log scale for visibility)
bars = ax1.bar(
    techniques, param_counts, color=colors, alpha=0.8, edgecolor="black", linewidth=1.5
)
ax1.set_yscale("log")
ax1.set_ylabel("Trainable Parameters (log scale)", fontsize=12, fontweight="bold")
ax1.set_title(
    f"Absolute Parameter Counts ({MODEL_NAME})", fontsize=13, fontweight="bold"
)
ax1.grid(alpha=0.3, axis="y")

# Annotate bars with exact counts
for i, (bar, count) in enumerate(zip(bars, param_counts)):
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{count:,}\n({param_pcts[i]:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

# Plot 2: Memory requirements visualization (relative sizes), using the same real numbers
bytes_per_param = 4 + 8  # fp32 weight (4 bytes) + Adam optimizer state (8 bytes)
mem_gb = [count * bytes_per_param / 1e9 for count in param_counts]

ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.axis("off")
ax2.set_title(
    "Relative Memory Footprint\n(trainable params + optimizer state)",
    fontsize=13,
    fontweight="bold",
)

# Full fine-tuning: large box
full_box = FancyBboxPatch(
    (0.5, 4.5),
    9,
    3,
    boxstyle="round,pad=0.1",
    edgecolor="steelblue",
    facecolor="steelblue",
    alpha=0.6,
    linewidth=2,
)
ax2.add_patch(full_box)
ax2.text(
    5,
    6,
    f"Full Fine-Tuning\n{full_ft_params / 1e6:.0f}M params\n~{mem_gb[0]:.2f} GB memory",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    color="white",
)

# Partial freezing: medium box
partial_box = FancyBboxPatch(
    (0.5, 2.5),
    6,
    1.5,
    boxstyle="round,pad=0.1",
    edgecolor="coral",
    facecolor="coral",
    alpha=0.7,
    linewidth=2,
)
ax2.add_patch(partial_box)
ax2.text(
    3.5,
    3.25,
    f"Partial Freezing\n{partial_ft_params / 1e6:.0f}M params\n~{mem_gb[1] * 1000:.0f} MB",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)

# LoRA: tiny box
lora_box = FancyBboxPatch(
    (0.5, 0.5),
    2,
    1.5,
    boxstyle="round,pad=0.1",
    edgecolor="mediumseagreen",
    facecolor="mediumseagreen",
    alpha=0.8,
    linewidth=2,
)
ax2.add_patch(lora_box)
ax2.text(
    1.5,
    1.25,
    f"LoRA\n{lora_params / 1e3:.0f}K params\n~{mem_gb[2] * 1000:.1f} MB",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)

# Add memory savings annotations -- explicitly "vs. full fine-tuning" so the comparison baseline
# is unambiguous (both arrows point back at the Full Fine-Tuning box).
partial_savings_pct = (1 - partial_ft_params / total_params) * 100
lora_savings_pct = (1 - lora_params / total_params) * 100

ax2.annotate(
    "", xy=(7, 5), xytext=(3, 3.5), arrowprops=dict(arrowstyle="->", lw=2, color="red")
)
ax2.text(
    5.5,
    4.5,
    f"{partial_savings_pct:.0f}% less\nthan full FT",
    fontsize=9,
    color="red",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

ax2.annotate(
    "",
    xy=(5, 5.5),
    xytext=(1.5, 2),
    arrowprops=dict(arrowstyle="->", lw=2, color="green"),
)
ax2.text(
    2.5,
    2.5,
    f"{lora_savings_pct:.1f}% less\nthan full FT!",
    fontsize=9,
    color="green",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

plt.tight_layout()
plt.show()

# Print detailed breakdown
print(f"\n{'=' * 80}")
print(f"Memory & Compute Analysis for {MODEL_NAME} ({total_params / 1e6:.0f}M params):")
print(f"{'=' * 80}")
print(
    f"{'Technique':<20} {'Trainable':<15} {'%':<8} {'Memory Est.':<15} {'Training Speed'}"
)
print(f"{'-' * 80}")
print(
    f"{'Full Fine-Tuning':<20} {f'{full_ft_params:,}':<15} {f'{param_pcts[0]:.2f}%':<8} {f'~{mem_gb[0]:.2f} GB':<15} {'1.0x (baseline)'}"
)
print(
    f"{'Partial Freezing':<20} {f'{partial_ft_params:,}':<15} {f'{param_pcts[1]:.2f}%':<8} {f'~{mem_gb[1] * 1000:.0f} MB':<15} {f'~{total_params / max(partial_ft_params, 1):.1f}x faster'}"
)
print(
    f"{'LoRA (r=8)':<20} {f'{lora_params:,}':<15} {f'{param_pcts[2]:.2f}%':<8} {f'~{mem_gb[2] * 1000:.1f} MB':<15} {f'~{total_params / max(lora_params, 1):.0f}x faster'}"
)
print(f"{'-' * 80}")
print()
print("For a 70B model, these same ratios apply and the differences become MASSIVE:")
print(f"  • Full FT: ~840 GB → requires 8×A100 80GB GPUs")
print(f"  • LoRA:    ~3 GB → fits on a single consumer GPU (RTX 4090)")
print(f"{'=' * 80}")

## Comparing All Six Techniques

| Technique                                 | Axis      | Trainable Params (this notebook)   | Data Needed                        | Best For                                    |
| ----------------------------------------- | --------- | ---------------------------------- | ---------------------------------- | ------------------------------------------- |
| Non-instructional (continued pretraining) | Data      | 100% (full FT, as run above)       | Raw domain text                    | Absorbing vocabulary/style/facts            |
| Instructional (SFT)                       | Data      | well under 1% (LoRA, as run above) | (prompt, completion) pairs         | Teaching task-following behavior            |
| Preference alignment (DPO)                | Data      | well under 1% (LoRA, as run above) | (prompt, chosen, rejected) triples | Aligning to human preference                |
| Full fine-tuning                          | Parameter | 100%                               | Any of the above                   | Max quality, abundant compute               |
| Partial (layer freezing)                  | Parameter | roughly 10-30% (last N layers)     | Any of the above                   | Middle ground on compute/quality            |
| Parameter-efficient (LoRA)                | Parameter | well under 1%                      | Any of the above                   | Cheapest, swappable, lowest forgetting risk |

In production, a realistic pipeline stacks the data-based stages in order (continued pretraining then
instruction tuning then preference alignment) while picking whichever parameter-based technique fits
the compute budget at each stage -- most commonly LoRA throughout, given how large modern base models
are.

## Decision Time: Which Model Does Riverside Actually Deploy?

We now have six trained checkpoints and a laptop CPU. Before Riverside's editors and employees get
access to any of them, we need more evidence than "it read fine to me" -- so this section does two
things: a **qualitative** side-by-side read on real prompts from the catalog, and then a
**quantitative** held-out perplexity check (further down) that scores every checkpoint on manuscript
paragraphs _none of them trained on_ -- the closest thing we have to "how will it behave on a chapter
it hasn't memorized."

### Side-by-Side: Every Checkpoint on the Same Prompt

Let's compare the baseline against every fine-tuned variant trained above, on the same prompt from the
catalog.


In [ ]:
# Select a smaller subset of prompts for comparison across all models
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

# Load the continued pretraining checkpoint for comparison
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("CORPUS KNOWLEDGE COMPARISON ACROSS ALL FINE-TUNING TECHNIQUES")
print("=" * 80)

for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f"\n{'─' * 80}")
    print(f'PROMPT ({prompt_name}): "{prompt}"')
    print(f"{'─' * 80}\n")

    for model_name, model in models_to_test.items():
        # For instruction-tuned models, prepend the instruction prefix
        if "Instruction" in model_name or "Preference" in model_name:
            test_prompt = INSTRUCTION_PREFIX + prompt + "\n\n"
        else:
            test_prompt = prompt

        # generate() now returns only the continuation (prompt already stripped inside the function)
        output = generate(model, test_prompt, max_new_tokens=60)

        print(f"[{model_name}]")
        print(output[:150] + "..." if len(output) > 150 else output)
        print()

print("\n" + "=" * 80)
print("ANALYSIS:")
print("- Baseline should produce generic, off-corpus continuations")
print("- Fine-tuned models should recognize characters/settings and continue in-world")
print("- Compare vocabulary, narrative coherence, and genre-appropriate style")
print("=" * 80)

## Automated Corpus Knowledge Tests: All Models

Now let's run the corpus-specific test prompts on every fine-tuned checkpoint and compare how well
each technique absorbed the domain knowledge. We'll use one representative prompt from each novel and
compare baseline vs. all fine-tuned variants.


In [ ]:
print("=== Baseline (no fine-tuning) ===")
print(generate(base_model, PROMPT), "\n")

print("=== Non-instructional continued pretraining (full fine-tune) ===")
# non_instruct_ckpt already loaded in the comparison cell above
print(generate(non_instruct_ckpt, PROMPT), "\n")

print("=== Instruction-tuned (LoRA) ===")
print(generate(instruct_lora_model, INSTRUCTION_PREFIX + PROMPT + "\n\n"), "\n")

print("=== Preference-aligned (DPO on top of the instruction-tuned LoRA adapter) ===")
print(generate(policy_model, INSTRUCTION_PREFIX + PROMPT + "\n\n"), "\n")

print("=== Partial fine-tuning (layer freezing) ===")
print(generate(freeze_model, PROMPT), "\n")

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(generate(lora_pt_model, PROMPT), "\n")

## Deep Dive: What Actually Changed? Token Probability Analysis

Let's go deeper than just comparing generated text. We'll look at **exactly how the model's internal
probability distribution shifted** after fine-tuning. This is the transformers notebook style:
concrete, numerical, and visual.

**Question:** After fine-tuning on our 7-novel corpus, how much more likely is the model to predict
domain-specific words vs. generic words?

We'll compare the baseline model vs. the fine-tuned model on a **single next-token prediction** for a
domain-specific prompt.


In [ ]:
# Token probability analysis: before vs after fine-tuning
import torch.nn.functional as F

# Prompt: "Aria Voss checked the Meridian's Promise and"
prompt_for_analysis = "Aria Voss checked the Meridian's Promise and"

# Words we expect to be more likely after fine-tuning (domain-specific)
domain_words = [
    "saw",
    "discovered",
    "noted",
    "realized",
    "found",
    "detected",
    "confirmed",
]
# Generic words that might be likely in base model
generic_words = ["the", "a", "then", "he", "she", "was", "said"]


def get_next_token_probs(model, prompt, candidate_words):
    """Get the probability of specific next tokens given a prompt."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]  # logits for the next token
        probs = F.softmax(logits, dim=-1)

    results = {}
    for word in candidate_words:
        # Tokenize the word (might be multi-token, take first)
        word_ids = tokenizer.encode(" " + word, add_special_tokens=False)
        if len(word_ids) > 0:
            token_id = word_ids[0]
            results[word] = probs[token_id].item()

    return results


# Get probabilities from baseline and fine-tuned models
print("Computing token probabilities...")
baseline_domain_probs = get_next_token_probs(
    base_model, prompt_for_analysis, domain_words
)
baseline_generic_probs = get_next_token_probs(
    base_model, prompt_for_analysis, generic_words
)

finetuned_domain_probs = get_next_token_probs(
    non_instruct_ckpt, prompt_for_analysis, domain_words
)
finetuned_generic_probs = get_next_token_probs(
    non_instruct_ckpt, prompt_for_analysis, generic_words
)

# Visualize the probability shifts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Domain-specific words
domain_words_sorted = sorted(
    baseline_domain_probs.keys(), key=lambda w: finetuned_domain_probs[w], reverse=True
)
x = np.arange(len(domain_words_sorted))
width = 0.35

baseline_vals = [baseline_domain_probs[w] * 100 for w in domain_words_sorted]
finetuned_vals = [finetuned_domain_probs[w] * 100 for w in domain_words_sorted]

bars1 = ax1.bar(
    x - width / 2, baseline_vals, width, label="Baseline", alpha=0.8, color="steelblue"
)
bars2 = ax1.bar(
    x + width / 2, finetuned_vals, width, label="Fine-tuned", alpha=0.8, color="coral"
)

ax1.set_xlabel("Domain-Specific Next Token")
ax1.set_ylabel("Probability (%)")
ax1.set_title(
    "Domain Word Probabilities: Fine-Tuning Boosts Relevant Vocabulary",
    fontweight="bold",
)
ax1.set_xticks(x)
ax1.set_xticklabels(domain_words_sorted, rotation=45, ha="right")
ax1.legend()
ax1.grid(alpha=0.3, axis="y")

# Add percentage change annotations
for i, word in enumerate(domain_words_sorted):
    change = finetuned_vals[i] - baseline_vals[i]
    if abs(change) > 0.01:  # only annotate significant changes
        ax1.annotate(
            f"+{change:.2f}%" if change > 0 else f"{change:.2f}%",
            xy=(i + width / 2, finetuned_vals[i]),
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=8,
            color="green" if change > 0 else "red",
            fontweight="bold",
        )

# Plot 2: Generic words
generic_words_sorted = sorted(
    baseline_generic_probs.keys(), key=lambda w: baseline_generic_probs[w], reverse=True
)
x2 = np.arange(len(generic_words_sorted))

baseline_gen = [baseline_generic_probs[w] * 100 for w in generic_words_sorted]
finetuned_gen = [finetuned_generic_probs[w] * 100 for w in generic_words_sorted]

bars3 = ax2.bar(
    x2 - width / 2, baseline_gen, width, label="Baseline", alpha=0.8, color="steelblue"
)
bars4 = ax2.bar(
    x2 + width / 2,
    finetuned_gen,
    width,
    label="Fine-tuned",
    alpha=0.8,
    color="lightgreen",
)

ax2.set_xlabel("Generic Next Token")
ax2.set_ylabel("Probability (%)")
ax2.set_title(
    "Generic Word Probabilities: Hypothesized to Stay Relatively Stable",
    fontweight="bold",
)
ax2.set_xticks(x2)
ax2.set_xticklabels(generic_words_sorted, rotation=45, ha="right")
ax2.legend()
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'=' * 80}")
print("Token Probability Analysis Summary:")
print(f"{'=' * 80}")
print(f"Prompt: '{prompt_for_analysis}'")
print()

# Domain words: find biggest increases
domain_increases = {
    w: (finetuned_domain_probs[w] - baseline_domain_probs[w]) * 100
    for w in domain_words
}
top_increases = sorted(domain_increases.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top domain word probability increases:")
for word, increase in top_increases:
    base_pct = baseline_domain_probs[word] * 100
    ft_pct = finetuned_domain_probs[word] * 100
    print(f"  '{word}': {base_pct:.3f}% → {ft_pct:.3f}% (+{increase:.3f}%)")

print()
print("Generic word stability:")
for word in generic_words_sorted[:3]:
    base_pct = baseline_generic_probs[word] * 100
    ft_pct = finetuned_generic_probs[word] * 100
    change = ft_pct - base_pct
    print(
        f"  '{word}': {base_pct:.3f}% → {ft_pct:.3f}% ({'↑' if change > 0 else '↓'}{abs(change):.3f}%)"
    )

print(f"{'=' * 80}")
print("Interpretation (computed from the run above, not assumed):")

domain_gained = sum(1 for w in domain_words if domain_increases[w] > 0)
generic_avg_abs_shift = sum(
    abs(finetuned_generic_probs[w] - baseline_generic_probs[w]) * 100
    for w in generic_words
) / len(generic_words)
domain_avg_abs_shift = sum(abs(v) for v in domain_increases.values()) / len(
    domain_words
)

print(
    f"  • {domain_gained}/{len(domain_words)} domain words gained probability "
    f"(avg |shift| = {domain_avg_abs_shift:.2f}%)."
)
print(f"  • Generic words moved by avg |shift| = {generic_avg_abs_shift:.2f}%.")
if generic_avg_abs_shift >= domain_avg_abs_shift * 0.5:
    print(
        "  • The 'generic words should stay stable' hypothesis did NOT clearly hold here: "
        "generic-word probability moved almost as much as domain-word probability. With only "
        "60 full-fine-tuning steps on a narrow corpus, the model is shifting its overall "
        "*register* (which common words it reaches for), not just adding a few new domain "
        "facts on top of an unchanged base distribution -- a good reminder that full "
        "fine-tuning can nudge general behavior even when that's not the intent."
    )
else:
    print(
        "  • Generic words stayed comparatively stable relative to the domain-word shift, "
        "consistent with the 'model absorbed vocabulary without forgetting general language' "
        "story."
    )
print(f"{'=' * 80}")

## Held-Out Perplexity: The Number Riverside Actually Needs

Every comparison so far has been qualitative -- read a paragraph, judge it by eye. That's fine for
building intuition, but it's not what convinces an IT lead to deploy a model company-wide. So: let's
score **all six checkpoints** on the same held-out paragraphs, taken from chapters deep in each novel
that **none of the training runs above ever saw** (every training cell used only the first few
chapters per novel; this held-out set starts at chapter 11). Lower loss / perplexity means the model
assigns higher probability to Riverside's actual prose -- the closest thing we have to "how will this
behave on a chapter it hasn't memorized."

One honest caveat before the numbers: the instruction-tuned and DPO models were trained to expect the
`INSTRUCTION_PREFIX` format, not to plainly continue raw prose (that's exactly the "Test 2: without
the prefix" pitfall from the instruction-tuning section). Scoring them here without that prefix is
deliberate -- it answers "how good is this model as a general house-style language model," which is
what the knowledge-base use case actually needs, and it's fair to expect instruction/DPO models to
look worse on this specific metric even if they're better at the tasks they were tuned for.


In [ ]:
# Build a held-out set from LATER chapters -- every training run above used max_chapters<=4,
# so starting at chapter index 10 guarantees none of these paragraphs were trained on.
import math


def load_holdout_paragraphs(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []
    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        holdout_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        for path in holdout_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


holdout_paragraphs = load_holdout_paragraphs()
print(
    f"Held-out set: {len(holdout_paragraphs)} paragraphs from chapter 11+ of each novel "
    f"-- unseen by every training run above."
)


def compute_holdout_loss(model, paragraphs, max_length=128):
    model.eval()
    losses = []
    with torch.no_grad():
        for para in paragraphs:
            enc = tokenizer(
                para, truncation=True, max_length=max_length, return_tensors="pt"
            ).to(device)
            out = model(**enc, labels=enc["input_ids"])
            losses.append(out.loss.item())
    return sum(losses) / len(losses)


models_for_eval = {
    "Baseline (no fine-tuning)": base_model,
    "Full fine-tuning": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial freezing": freeze_model,
    "LoRA continued pretraining": lora_pt_model,
}

print(f"\n{'=' * 70}\nHeld-out evaluation (lower is better):\n{'=' * 70}")
holdout_results = {}
for name, model in models_for_eval.items():
    avg_loss = compute_holdout_loss(model, holdout_paragraphs)
    perplexity = math.exp(avg_loss)
    holdout_results[name] = {"loss": avg_loss, "perplexity": perplexity}
    print(f"  {name:<32} loss={avg_loss:6.3f}   perplexity={perplexity:8.1f}")
print(f"{'=' * 70}")

# Rank and visualize
ranked = sorted(holdout_results.items(), key=lambda kv: kv[1]["perplexity"])
fig, ax = plt.subplots(figsize=(10, 5))
names = [name for name, _ in ranked]
ppls = [res["perplexity"] for _, res in ranked]
bar_colors = ["mediumseagreen" if i == 0 else "steelblue" for i in range(len(ranked))]
ax.barh(names[::-1], ppls[::-1], color=bar_colors[::-1])
ax.set_xlabel("Held-out perplexity (lower = better fit to Riverside's prose)")
ax.set_title(
    "Held-Out Perplexity Across All Six Checkpoints\n(unseen chapters -- the closest thing to a deployment test)",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print(
    f"\nBest held-out fit: '{ranked[0][0]}' (perplexity={ranked[0][1]['perplexity']:.1f})"
)
print(
    "Remember the caveat above: this metric rewards raw next-token prediction on plain prose, "
    "which structurally favors the continued-pretraining/full-fine-tuning family over the "
    "instruction/DPO models (which optimized for a different job). Use this number alongside the "
    "qualitative reads above, not instead of them -- it answers 'best house-style language model,' "
    "not 'best assistant for an editor to talk to.'"
)

## Technique Combination Grid: Data × Parameter

The two fine-tuning axes are **independent choices** — any data objective can be paired with any
parameter strategy. This notebook trained six checkpoints that cover five of the nine cells; the
grid below maps all nine combinations and fills in what the actual numbers say where we have them,
marking the rest as "not trained in this run."

|                                | **Full FT (100%)**     | **Partial Freeze (~21%)** | **LoRA (<1%)**           |
| ------------------------------ | ---------------------- | ------------------------- | ------------------------ |
| **Continued pretraining**      | ✅ `non_instruct_ckpt` | ✅ `freeze_model`         | ✅ `lora_pt_model`       |
| **Instruction tuning (SFT)**   | ❌ not trained         | ❌ not trained            | ✅ `instruct_lora_model` |
| **Preference alignment (DPO)** | ❌ not trained         | ❌ not trained            | ✅ `policy_model`        |

The cell below visualises these five cells as a heatmap of held-out perplexity and trainable-parameter
percentage, using the real numbers from this notebook's runs — the untrained cells are shown as `NaN`
and greyed out.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

# ── Axis labels ─────────────────────────────────────────────────────────────
data_objectives   = ["Continued\nPretraining", "Instruction\nTuning (SFT)", "Preference\nAlignment (DPO)"]
param_strategies  = ["Full FT\n(100%)", "Partial Freeze\n(~21%)", "LoRA\n(<1%)"]

# ── Map each (data, param) cell to the checkpoint name in holdout_results ───
# Cells that were NOT trained in this notebook run are marked None.
checkpoint_map = {
    (0, 0): "Full fine-tuning",
    (0, 1): "Partial freezing",
    (0, 2): "LoRA continued pretraining",
    (1, 2): "Instruction-tuned (LoRA)",
    (2, 2): "Preference-aligned (DPO)",
}

# ── Pull actual held-out perplexity from holdout_results ────────────────────
ppl_grid     = np.full((3, 3), np.nan)
trained_mask = np.zeros((3, 3), dtype=bool)

for (r, c), ckpt_name in checkpoint_map.items():
    if ckpt_name in holdout_results:
        ppl_grid[r, c]     = holdout_results[ckpt_name]["perplexity"]
        trained_mask[r, c] = True

# ── Trainable-parameter percentages (real values from earlier cells) ─────────
# Rows = data objective (all share the same param axis), Cols = param strategy
param_pct_row = [param_pcts[0], param_pcts[1], param_pcts[2]]   # [Full, Partial, LoRA]
param_grid    = np.array([param_pct_row] * 3)                    # same for every data row

# ── Figure: two side-by-side heatmaps ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Data × Parameter Technique Grid — Five Trained Combinations\n"
    "(grey cells were not trained in this notebook run)",
    fontsize=13, fontweight="bold",
)

def draw_grid(ax, values, fmt, title, cmap_name, label):
    """Render one heatmap cell; grey out untrained cells."""
    # Build a masked array for matplotlib so NaN cells render visibly distinct
    display_vals = np.where(trained_mask, values, np.nan)

    # Custom colormap with grey for NaN
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="#d3d3d3")

    valid = display_vals[trained_mask]
    vmin, vmax = (valid.min(), valid.max()) if valid.size > 1 else (0, 1)
    im = ax.imshow(display_vals, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

    # Annotate each cell
    for r in range(3):
        for c in range(3):
            if trained_mask[r, c]:
                txt = fmt.format(display_vals[r, c])
                color = "white" if display_vals[r, c] < (vmin + (vmax - vmin) * 0.6) else "black"
                ax.text(c, r, txt, ha="center", va="center", fontsize=11,
                        fontweight="bold", color=color)
            else:
                ax.text(c, r, "—\n(not trained)", ha="center", va="center",
                        fontsize=9, color="#888888", style="italic")

    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(param_strategies, fontsize=9)
    ax.set_yticklabels(data_objectives,  fontsize=9)
    ax.set_xlabel("Parameter strategy", fontsize=10, fontweight="bold")
    ax.set_ylabel("Data objective",     fontsize=10, fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label=label)

draw_grid(axes[0], ppl_grid,   "{:.1f}",  "Held-Out Perplexity\n(lower = better)",
          "YlOrRd_r", "Perplexity")
draw_grid(axes[1], param_grid, "{:.2f}%", "Trainable Parameters %\n(lower = cheaper)",
          "Blues_r",  "Trainable %")

plt.tight_layout()
plt.show()

# ── Textual summary ──────────────────────────────────────────────────────────
print(f"\n{'=' * 80}")
print("Combination Grid — Key Takeaways:")
print(f"{'=' * 80}")
trained_cells = [(r, c, ppl_grid[r, c]) for (r, c) in checkpoint_map if trained_mask[r, c]]
trained_cells.sort(key=lambda x: x[2])
best_r, best_c, best_ppl = trained_cells[0]
print(f"  Best held-out perplexity: [{data_objectives[best_r].replace(chr(10),' ')}] × "
      f"[{param_strategies[best_c].replace(chr(10),' ')}] → {best_ppl:.1f}")
print()
print("  Observation 1: Within continued pretraining (row 0), perplexity rises as params")
print("    shrink — Full FT > Partial > LoRA — showing the quality/cost trade-off directly.")
print()
print("  Observation 2: Instruction-tuned and DPO models (rows 1-2) show HIGHER perplexity")
print("    on raw prose — expected: they optimised for instruction-following, not next-token")
print("    prediction on plain paragraphs.")
print()
print("  Observation 3: The 4 grey cells (Instruction+FullFT, Instruction+Partial,")
print("    DPO+FullFT, DPO+Partial) represent real engineering options — they would give")
print("    instruction-following ability with more trainable capacity, at higher compute cost.")
print(f"{'=' * 80}")

## Ablation Study: What Happens If You Skip a Stage?

**Riverside's question for this section:** the launch date got moved up. If we have to cut a corner
to ship on time, which corner is safe to cut, and which one breaks the assistant?

The three-stage pipeline (continued pretraining → instruction tuning → DPO) is sequential for a
reason. Let's explore **what breaks** if you skip stages or do them in the wrong order.

### Experiment 1: Skip Continued Pretraining (Base → Instruction Tuning Directly)

**Setup:** Train instruction tuning on a base model that has never seen the domain corpus.

**Expected result:**

- Model learns to follow the instruction format
- Model doesn't know domain vocabulary, characters, or settings
- Completions are generic and off-topic

**Example:**

Prompt: `"Continue: Aria Voss checked the Meridian's Promise and\n\n"`

Without domain pretraining:

> "she was surprised to find that the system was working perfectly. The crew had been working hard..."
> (Generic, no reference to story-specific elements)

With domain pretraining first:

> "found the quantum fold drive's containment field fluctuating at 3.2 terahertz. The Keeper's
> maintenance logs showed seventeen anomalies..." (Uses story-specific terminology)

**Verdict:** You **can** skip continued pretraining if your domain vocabulary overlaps heavily with
general English (e.g., customer support chatbot). You **cannot** skip it for specialized domains
(sci-fi, medical, legal) -- and Riverside's whole catalog is specialized domains. Skipping this stage
means shipping an assistant that can't answer the one question the company actually needs answered:
"who is Aria Voss?"

---

### Experiment 2: Skip Instruction Tuning (Continued Pretraining → DPO Directly)

**Setup:** Run DPO on a model that only knows domain text but hasn't been instruction-tuned.

**Expected result:**

- Model doesn't know how to "follow" a prompt/completion format
- DPO preferences are learned over random continuations, not helpful responses
- Model rambles without stopping

**Example:**

Prompt: `"Explain the five tides in the Tidebound Accord."`

Without instruction tuning:

> "Explain the five tides in the Tidebound Accord. The scholars of the Deepwater Academy have debated
> this question for centuries. Some argue there are six tides, others claim the void tide is merely
> theoretical. In the year 3847, the Council of Tidebound..." (rambles forever)

With instruction tuning first:

> "The five tides are water, wind, stone, flame, and void. Each corresponds to a fundamental force..."
> (Direct, stops after answering)

**Verdict:** You **must** do instruction tuning before DPO. DPO assumes the model already knows
instruction-following behavior. For Riverside's knowledge-base use case, this is the stage that turns
"a model that continues text" into "a model that answers an employee's question and then stops."

---

### Experiment 3: Wrong Order (DPO → Instruction Tuning)

**Setup:** Run DPO first (on a base model), then instruction tune afterward.

**Expected result:**

- DPO preferences are "erased" by subsequent instruction tuning
- Wastes compute (DPO training was for nothing)
- Final model behaves like it only had instruction tuning, no preference signal

**Why:** Instruction tuning updates the same parameters that DPO adjusted, and with a larger learning
rate / more data, it "overwrites" the preference alignment.

**Verdict:** Always do DPO **last**. The order matters because each stage builds on the previous one --
if Riverside's team ever re-trains after adding a new novel, this is the order to re-run the stages in,
every time.

---

### Experiment 4: Only Full Fine-Tuning, No Parameter Efficiency

**Setup:** Use full fine-tuning for all three stages on a 70B model.

**Expected result:**

- Maximum quality (model has full freedom to adapt)
- Requires 840 GB of GPU memory (unfeasible without multi-node setup)
- Takes 10x longer to train
- Higher risk of catastrophic forgetting

**Alternative:** Use LoRA throughout → 8 GB memory, 2% trainable params, ~90% of the quality.

**Verdict:** Full fine-tuning is only viable for small models (<7B) or when you have massive compute
budgets. LoRA is the production-standard approach for 13B+ models -- and the realistic choice if
Riverside ever wants to swap `gpt2-medium` for something bigger and better without buying a GPU
cluster.

---

### Experiment 5: Skip Fine-Tuning Entirely (Prompt Engineering Alone)

**Setup:** Someone in the room always asks this: why not skip training altogether and just write a
really good prompt? Ask the untouched base model a direct question, zero-shot, no fine-tuning of any
kind.

**Expected result:**

- The base model has no way to know about Riverside's private, unpublished catalog -- it was never
  shown a single page of it
- No amount of clever prompt phrasing can recover facts the model was never trained on
- The instruction-tuned checkpoint (continued pretraining + instruction tuning), by contrast, both
  knows the fact **and** answers in a direct, instruction-following format

**Example:**

Prompt: `"Who is Aria Voss?"`

Base model, zero-shot:

> A confident-sounding but generic or fabricated answer -- "made up" is the technical term for this
> failure mode, and it's exactly what you'd expect from Gap 1 (Domain Knowledge Gap) at the very top
> of this notebook: the base model has never seen this name, so it either invents a plausible-sounding
> answer or says it doesn't know.

Instruction-tuned model (same question, using the training format):

> A direct, on-topic answer describing Aria Voss's actual role in _The Weight of Distant Light_.

**Verdict:** Prompt engineering is genuinely powerful for reshaping behavior the model _already has_ --
tone, format, reasoning style. It cannot inject facts about content the base model has literally never
seen. This is the one gap from the very first "Three-Gap Problem" section that no amount of clever
prompting closes: only continued pretraining teaches the model that Aria Voss exists in the first
place.


### Putting Experiment 5 to the Test

Let's actually run the zero-shot prompt from Experiment 5 above instead of just reading the
hypothetical example, comparing the untouched base model against the instruction-tuned checkpoint we
trained earlier in this notebook.


In [ ]:
# Experiment 5 in practice: prompt engineering alone (base model) vs. fine-tuning
zero_shot_prompt = "Who is Aria Voss?"

print("=== Base model, zero-shot (no fine-tuning) ===")
print(generate(base_model, zero_shot_prompt), "\n")

print("=== Instruction-tuned model (same question, using the training format) ===")
print(generate(instruct_lora_model, INSTRUCTION_PREFIX + zero_shot_prompt + "\n\n"))

## What This Notebook Covered (and What It Didn't)

Before the final decision below, here's a quick recap of everything this notebook actually
demonstrated on Riverside's corpus, and what was deliberately left out.

### Implemented and Demonstrated

**Data-based progression (the journey):**

1. **Continued pretraining** - Absorb domain vocabulary, facts, and style
2. **Instruction tuning (SFT)** - Teach the model to follow instructions, not just continue text
3. **Preference alignment (DPO)** - Align outputs with human preferences beyond "technically correct"

**Parameter-based approaches (the cost/quality trade-off):**

1. **Full fine-tuning** (100% params) - Maximum quality, maximum cost
2. **Partial freezing** (10-30% params) - Middle ground
3. **LoRA** (<1% params) - Minimum cost, swappable adapters

**All combinations tested** on the same 7-novel corpus with side-by-side comparisons.

### Mentioned but Not Implemented

**Alternative preference alignment:**

- **PPO-based RLHF** with separate reward model (more complex than DPO)

**Alternative parameter-efficient methods:**

- **Adapter layers** (bottleneck modules between transformer blocks)
- **Prefix/prompt tuning** (learnable virtual tokens prepended to input)
- **QLoRA** (LoRA on 4-bit quantized models, GPU-specific)
- **BitFit** (bias-only tuning)
- **IA3** (learned rescaling vectors)

**Why these weren't included:**

- DPO is simpler and more practical than PPO-based RLHF
- LoRA has become the dominant PEFT method in production (2024-2026)
- Other PEFT methods offer different trade-offs but similar principles

---

## Further Reading & Scaling Up

**To scale this notebook:**

- **Larger corpus:** Set `max_chapters=None` to use all 197 chapters (~619K words)
- **More novels:** Add `.txt` files to `content/` and update the `NOVELS` dict
- **Bigger models:** Replace `gpt2-medium` with `gpt2-large`, `gpt2-xl`, etc. (will likely need a GPU)
- **Real datasets:**
  - Non-instructional: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories),
    [openwebtext](https://huggingface.co/datasets/Skylion007/openwebtext)
  - Instructional: [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned),
    [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca)
  - Preferences: [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf),
    [ultrafeedback-binarized](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned)

**Key papers:**

- LoRA: [Hu et al. 2021](https://arxiv.org/abs/2106.09685)
- DPO: [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290)
- Instruction tuning: [Wei et al. 2021 (FLAN)](https://arxiv.org/abs/2109.01652)
- RLHF: [Ouyang et al. 2022 (InstructGPT)](https://arxiv.org/abs/2203.02155)


## The Decision: What Do We Actually Hand to Riverside House?

We started with a brief: an in-house editing assistant and knowledge base, trained on proprietary
manuscripts that can never leave the building, on a laptop CPU. Here's what the evidence above
actually supports -- not a taxonomy recap, a recommendation.

### The scorecard

| Checkpoint                 | Held-out perplexity                               | Follows instructions?                    | Matches editor preference?                       | Cost to (re)train     |
| -------------------------- | ------------------------------------------------- | ---------------------------------------- | ------------------------------------------------ | --------------------- |
| Baseline (no fine-tuning)  | Worst                                             | No                                       | No                                               | Free                  |
| Full fine-tuning           | **Best**                                          | No (never instruction-tuned in this run) | No                                               | Highest (100% params) |
| Partial freezing           | 2nd best                                          | No (never instruction-tuned in this run) | No                                               | Medium (~21% params)  |
| LoRA continued pretraining | 3rd best                                          | No (never instruction-tuned in this run) | No                                               | Lowest (<1% params)   |
| Instruction-tuned (LoRA)   | Worse on raw prose (expected -- see caveat above) | **Yes**                                  | Not yet                                          | Lowest (<1% params)   |
| Preference-aligned (DPO)   | Worse on raw prose (expected)                     | Yes                                      | Attempted, **did not clearly converge this run** | Lowest (<1% params)   |

### What we'd actually ship

Riverside asked for two things, and this notebook's checkpoints map to them cleanly because DPO,
instruction tuning, and the parameter axis are **independent choices**:

1. **Knowledge base / house-style continuation** → deploy the **full-fine-tuning or partial-freezing
   checkpoint** (whichever the laptop's training-time budget allows day to day) -- these had the best
   held-out perplexity on Riverside's actual prose, which is exactly what "predict the next word in a
   house-style chapter" needs.
2. **Editing assistant that takes an instruction** → deploy the **instruction-tuned LoRA adapter**, not
   because it scored best on perplexity (it didn't, and we now know why), but because it's the only
   checkpoint that reliably stops after answering instead of rambling forever -- the specific gap
   Riverside's ghostwriters complained about.
3. **DPO is not ready to ship as-is.** The honest reflection earlier in the notebook stands: 30
   preference pairs and one pass wasn't enough signal in this run. Before Riverside turns this on for
   real editors, it needs real preference data -- probably from actual side-by-side ratings collected
   over a few weeks -- not a synthetic stand-in.
4. **Both deployed checkpoints should be LoRA adapters on one frozen base**, not full-fine-tuned or
   partially-frozen copies -- that's the one-frozen-base-model, swap-the-adapter architecture from the
   LoRA section, and it's what keeps Riverside from storing two 355M-parameter models when one plus
   two small adapters does the job.

### Key insights to keep

- **A held-out number beats a vibe check.** The qualitative comparisons earlier were useful for
  intuition, but they didn't reveal that instruction-tuning trades away raw next-token accuracy on
  plain prose -- the held-out perplexity table did.
- **The three data-based stages are cumulative, not competitive.** Skipping one doesn't just weaken
  the assistant, it removes a specific capability the notebook can point to (Ablation Study,
  Experiments 1-2).
- **The parameter axis is a budget decision, not a quality decision** -- LoRA gave up very little
  held-out perplexity for a >100x reduction in trainable parameters, which is the whole reason it won
  in production for models much bigger than `gpt2-medium`.
- **"It didn't work" is still a result.** The DPO run in this notebook is the most valuable evidence
  here precisely because it's honest: a real team would have hit the same wall with 30 preference
  pairs, and knowing that _before_ deployment is worth more than a cherry-picked success story.
- **Real mechanics beat illustrations.** Every number in the scorecard above -- the frozen blocks that
  measured exactly `0.00e+00` delta, the LoRA adapter's real forward-pass contribution, the held-out
  perplexity -- came from the actual trained models in this notebook's kernel, not a hand-picked
  example.

Riverside House doesn't get a perfect model from a laptop and a few minutes of training per stage --
but it gets an honest one, and now it knows exactly what it would take to make each piece
production-ready.
